tb_estoque_principal_geral_62

tb_estoque_resultado_62

In [1]:
/* ===  ACESSO POSTGRESQL ESTOQUE METODOLOGIA 62 === */
PROC SQL;
  RESET EXEC;

  CONNECT TO POSTGRES AS mypg
    ( SERVER   = "172.16.14.11"
      PORT     = 5432
      DATABASE = "curadoria"
      USER     = "painel_user"
      PASSWORD = "painel_user"
      READBUFF = 10000
      CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  /* =========================================================
     CONSULTA 1: BASE gp + phv (FILTRA phv.numero IS NOT NULL)
     ========================================================= */
  EXECUTE
  (
    DROP TABLE IF EXISTS tmp_base_painel;

    CREATE TEMP TABLE tmp_base_painel AS
    SELECT
          gp.nom,
          gp.cd_cli,
          gp.tipo,
          gp.pilar,
          gp.atual,
          gp.ts_execucao,
          gp.ts_aclt_slc,
          gp.nr_seql_slct,
          gp.nr_seql_pecs_nvl,
          phv.entrada,
          phv.protocolo_ano,
          phv.protocolo_num
    FROM HIGHVAREJO.GERENCIAMENTO_painel gp
    INNER JOIN HIGHVAREJO.protocolo phv
            ON phv.cd_cli           = gp.cd_cli
           AND phv.ts_aclt_slc      = gp.ts_aclt_slc
           AND phv.nr_seql_slct     = gp.nr_seql_slct
           AND phv.nr_seql_pecs_nvl = gp.nr_seql_pecs_nvl
           AND phv.ts_execucao      = gp.ts_execucao
    WHERE phv.numero IS NOT NULL
  ) BY mypg;

  /* =========================================================
     CONSULTA 2: SELECT FINAL (JOIN sa/dds/fun) + FILTRO sa
     ========================================================= */
  CREATE TABLE work.inicial_painel AS
    SELECT *
    FROM CONNECTION TO mypg
    (
      SELECT DISTINCT
            0 AS id, /* id fixo sempre 0 */
            CONCAT(gp.protocolo_ano::text, LPAD(gp.protocolo_num::text, 8, '0')) AS protocolo, /* protocolo: ano + número com 8 dígitos */
            gp.nom AS nome_cliente,
            gp.cd_cli AS cod_mci,
            gp.entrada AS dta_entrada,
            NULL::date AS dta_saida, /* dta_saida fixa vazia */
            DATE(sa.ts_execucao) AS dta_ini_etapa,
            CASE
                WHEN sa.status IS NULL THEN 'AGUARDANDO ANÁLISE' /* Aguardando atribuição */
                WHEN sa.status = 1 THEN 'AGUARDANDO ANÁLISE' /* Aguardando análise */
                WHEN sa.status IN (-1, -2, 2) THEN 'EM ANÁLISE' /* Devolvido para alterações Em análise */
                WHEN sa.status IN (3, 4) THEN 'COMITÊ' /* Aguardando 1° voto - Aguardando 2° voto */
                WHEN sa.status IN (5, 95) THEN 'EM ENCERRAMENTO' /* Em processamento */
                WHEN sa.status IN (6, 98) THEN 'FINALIZADO' /* Processado Devolvido */
                ELSE sa.nome_status
            END AS etapa,
            CASE
                WHEN sa.status = -1 THEN 'DEVOLVIDO'
                WHEN sa.status = -2 THEN 'DEVOLVIDO'
                WHEN sa.status = 98 THEN 'DEVOLVIDO'
                WHEN sa.status = 6 THEN 'CONCLUÍDO'
                ELSE ''
            END AS tp_finalizacao,
            UPPER(gp.tipo) AS tipo_processo,
            'LIMITE DE CRÉDITO-PJ-METOD.ANALITICA' AS tipo_produto, 
            'GEREM' AS gerencia,
            'P. JURÍDICA' AS tipo_cliente,
            UPPER(gp.pilar) AS pilar,
            'GEREM-SP ANALÍTICO' AS unidade_analise, /* GEREM-SP ANALÍTICO REUNIÃO COM LYGIA */
            dds.cd_depe_rsp_anl AS prefixo_agencia,
            fun.nome AS funci_etapa,
            'GEREM-SP DIV 5 PJ 62' AS equipe, /* NOME DO GERENTE ################################# #################### REVISAR */
            CASE
                WHEN gp.pilar = 'Atacado' THEN 'LYGIA CUNHA'
                ELSE 'ALBANO CARVALHO'
            END AS gerente,
            dds.dt_vnct_lim AS dta_vencimento_limite,
            gp.atual AS vlr_limite_atual,

            '' AS vlr_limite_solicitado, /* ###################### ################################# #################### REVISAR */

            DATE(gp.entrada + INTERVAL '10 days') AS dta_pacto, /* REUNIÃO COM LYGIA */
            CURRENT_DATE AS dta_estoque,

            /* dias_uteis_dm - Dias úteis entre a entrada e a data de hoje */

            (CURRENT_DATE - DATE(gp.entrada)) AS dias_corridos_dm,

            /* dias_uteis_pacto - Dias úteis entre o pacto e a data de hoje */

            ((DATE(gp.entrada) + 15) - CURRENT_DATE) AS dias_corridos_pacto,

            /* dias_uteis_etapa - Dias úteis entre o início da etapa e a data de hoje */

            (CURRENT_DATE - DATE(sa.ts_execucao)) AS dias_corridos_etapa,

            /* dias_uteis_dm - 0_4d: flag 0 a 4 dias úteis = TRUE */
            /* 5_9d: flag 5 a 9 dias úteis = TRUE */
            /* 10_14d: flag 10 a 14 dias úteis = TRUE */
            /* 15_19d: flag 15 a 19 dias úteis = TRUE */
            /* 20_24d: flag 20 a 24 dias úteis = TRUE */
            /* 25_29d: flag 25 a 29 dias úteis = TRUE */
            /* 30_39d: flag 30 a 39 dias úteis = TRUE */
            /* 40_49d: flag 40 a 49 dias úteis = TRUE */
            /* 50_59d: flag 50 a 59 dias úteis = TRUE */
            /* 60_d: flag 60 dias úteis ou mais = TRUE */

            'USO' AS tipo_dd,
            NULL::date AS dt_ini_triagem,
            NULL::date AS dt_fim_triagem,
            NULL::date AS dt_ini_diligencia,
            NULL::date AS dt_fim_diligencia,
            NULL::date AS dt_ini_esc_superior,
            NULL::date AS dt_fim_esc_superior,
            NULL::date AS tm_grp,
            sa.chave_responsavel_atual AS matricula_funci_etapa,

            CASE
                WHEN gp.pilar = 'Atacado' THEN 'F6623739' /* 'LYGIA CUNHA' */
                ELSE 'F0281079' /* 'ALBANO CARVALHO' */
            END AS matricula_gerente,

            NULL::date AS dt_ini_ag_analise,
            NULL::date AS dt_fim_ag_analise,

            gp.protocolo_ano,
            gp.protocolo_num,
            gp.cd_cli,
            gp.ts_aclt_slc,
            gp.nr_seql_slct,
            gp.nr_seql_pecs_nvl

      FROM tmp_base_painel gp

      LEFT JOIN HIGHVAREJO.status_atual sa
             ON sa.cd_cli           = gp.cd_cli
            AND sa.ts_aclt_slc      = gp.ts_aclt_slc
            AND sa.nr_seql_slct     = gp.nr_seql_slct
            AND sa.nr_seql_pecs_nvl = gp.nr_seql_pecs_nvl

      LEFT JOIN PUBLIC.dds_anl_hv dds
            ON dds.cd_cli           = gp.cd_cli
           AND dds.ts_aclt_slc      = gp.ts_aclt_slc
           AND dds.nr_seql_slct     = gp.nr_seql_slct
           AND dds.nr_seql_pecs_nvl = gp.nr_seql_pecs_nvl
           AND dds.ts_execucao      = gp.ts_execucao

      LEFT JOIN DMTV_GEREM.funcionarios fun
            ON fun.chave            = sa.chave_responsavel_atual

      WHERE
        /* 6 - Processado - 98 - Devolvido - 99 - Cancelada */
        (sa.status NOT IN (6, 98, 99) OR sa.status IS NULL)
        /* AND gp.cd_cli = 209619444 */

    );

  DISCONNECT FROM mypg;
QUIT;

PROC PRINT DATA=work.inicial_painel(OBS=3);
RUN;

Please enter the name of the SAS Config you wish to run. Available Configs are: ['sasapp_anl02', 'sasapp_anl03', 'sasapp_anl04', 'sasapp_anl05', 'sasapp_anl06', 'sasapp_anl07', 'sasapp_anl08', 'sasapp_anl09', 'sasapp_anl10', 'sasapp_anl11', 'sasapp_anl12', 'sasapp_anl13']  sasapp_anl02
Please enter the OMR user id:  f1021796
Please enter the password for OMR user :  ········


SAS Connection established. Subprocess id is 2557



Obs,id,protocolo,nome_cliente,cod_mci,dta_entrada,dta_saida,dta_ini_etapa,etapa,tp_finalizacao,tipo_processo,tipo_produto,gerencia,tipo_cliente,pilar,unidade_analise,prefixo_agencia,funci_etapa,equipe,gerente,dta_vencimento_limite,vlr_limite_atual,vlr_limite_solicitado,dta_pacto,dta_estoque,dias_corridos_dm,dias_corridos_pacto,dias_corridos_etapa,tipo_dd,dt_ini_triagem,dt_fim_triagem,dt_ini_diligencia,dt_fim_diligencia,dt_ini_esc_superior,dt_fim_esc_superior,tm_grp,matricula_funci_etapa,matricula_gerente,dt_ini_ag_analise,dt_fim_ag_analise,protocolo_ano,protocolo_num,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl
1,0,202600280586,RURAL TECH COMERCIO E SERVICOS LTDA,445064998,04MAR2026,.,10MAR2026,COMITÊ,,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,GEREM,P. JURÍDICA,VAREJO,GEREM-SP ANALÍTICO,452,PAULO LAVINIO DE ARAUJO JUNIOR,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,24JAN2027,1000000.00,,14MAR2026,12MAR2026,8,7,2,USO,.,.,.,.,.,.,.,F8125991,F0281079,.,.,2026,280586,445064998,03MAR2026:16:28:53.850110,1,2
2,0,202600282205,SUPERMERCADO VENEZA LTDA - EPP,380530,06MAR2026,.,10MAR2026,COMITÊ,,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,GEREM,P. JURÍDICA,VAREJO,GEREM-SP ANALÍTICO,773,PAULO LAVINIO DE ARAUJO JUNIOR,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,18FEB2026,1000000.00,,16MAR2026,12MAR2026,6,9,2,USO,.,.,.,.,.,.,.,F8125991,F0281079,.,.,2026,282205,380530,05MAR2026:16:07:01.491253,1,2
3,0,202600282206,PROPILEU SANEAMENTO E CONSTRUCOES LTDA.,32300523,06MAR2026,.,10MAR2026,COMITÊ,,RECONSID,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,GEREM,P. JURÍDICA,VAREJO,GEREM-SP ANALÍTICO,386,PAULO LAVINIO DE ARAUJO JUNIOR,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,25FEB2027,3020000.00,,16MAR2026,12MAR2026,6,9,2,USO,.,.,.,.,.,.,.,F8125991,F0281079,.,.,2026,282206,32300523,25FEB2026:16:36:26.365501,2,2


In [2]:
/* === TABELA CALENDÁRIO - LISTA DIAS ÚTEIS E NÃO ÚTEIS NO PERÍODO === */
DATA work.base_datas;
    FORMAT data_ref YYMMDD10.;
    DO data_ref = '01JAN2020'd TO '31DEC2036'd;
        OUTPUT;
    END;
RUN;

/* Gera colunas de dias da semana e flag de dia útil */
PROC SQL;
    CREATE TABLE work.calendario AS
    SELECT 
        data_ref,
        weekday(data_ref) AS dia_semana,
        
        CASE 
            /* 1. Exclui Finais de Semana (1=Domingo, 7=Sábado) */
            WHEN weekday(data_ref) IN (1, 7) THEN 0
            
            /* 2. Exclui Feriados Fixos (Nacionais + SP Capital) */
            WHEN month(data_ref) = 1 AND day(data_ref) = 1 THEN 0   /* Ano Novo */
            WHEN month(data_ref) = 1 AND day(data_ref) = 25 THEN 0  /* Aniversário de SP */
            WHEN month(data_ref) = 4 AND day(data_ref) = 21 THEN 0  /* Tiradentes */
            WHEN month(data_ref) = 5 AND day(data_ref) = 1 THEN 0   /* Dia do Trabalho */
            WHEN month(data_ref) = 7 AND day(data_ref) = 9 THEN 0   /* Rev. Constitucionalista */
            WHEN month(data_ref) = 9 AND day(data_ref) = 7 THEN 0   /* Independência */
            WHEN month(data_ref) = 10 AND day(data_ref) = 12 THEN 0 /* N. Sra. Aparecida */
            WHEN month(data_ref) = 11 AND day(data_ref) = 2 THEN 0  /* Finados */
            WHEN month(data_ref) = 11 AND day(data_ref) = 15 THEN 0 /* Proc. da República */
            WHEN month(data_ref) = 11 AND day(data_ref) = 20 THEN 0 /* Consciência Negra */
            WHEN month(data_ref) = 12 AND day(data_ref) = 25 THEN 0 /* Natal */
            
            /* 3. Exclui Feriados Móveis (Baseados na função de Páscoa do SAS) */
            WHEN data_ref = holiday('EASTER', year(data_ref)) - 48 THEN 0 /* Segunda de Carnaval */
            WHEN data_ref = holiday('EASTER', year(data_ref)) - 47 THEN 0 /* Terça de Carnaval */
            WHEN data_ref = holiday('EASTER', year(data_ref)) - 2 THEN 0  /* Sexta-feira Santa */
            WHEN data_ref = holiday('EASTER', year(data_ref)) + 60 THEN 0 /* Corpus Christi */
            
            /* Dias normais de trabalho recebem a flag 1 */
            ELSE 1 
        END AS flg_dia_util

    FROM work.base_datas;
QUIT;

proc print data=work.calendario(obs=3);
run;

Obs,data_ref,dia_semana,flg_dia_util
1,2020-01-01,4,0
2,2020-01-02,5,1
3,2020-01-03,6,1


In [3]:
/* === PIVOT DA TABELA COM TIMESTAMP DE CADA FASE DO PROCESSO ===*/

DATA work.map_status;
  LENGTH cod_status 8 nome_col $40;
  INFILE datalines DSD TRUNCOVER;
  INPUT cod_status nome_col :$40.;
DATALINES;
1,dt_entrada
2,dt_ini_analise
3,dt_ini_comite
4,dt_ini_comite_segundo_voto
99,dt_cancelamento
5,dt_finalizacao
-1,dt_devolucao
-2,dt_devolucao2
6,dt_saida
98,dt_devolucao3
95,dt_finalizacao2
;
RUN;

PROC SQL;
  RESET EXEC;

  CONNECT TO postgres AS mypg
    (SERVER   = "172.16.14.11"
     PORT     = 5432
     DATABASE = "curadoria"
     USER     = "painel_user"
     PASSWORD = "painel_user"
     READBUFF = 10000
     CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE work.logs_gerenciamento AS
  SELECT *
  FROM CONNECTION TO mypg
  (
    SELECT *
    FROM HIGHVAREJO.LOGS_GERENCIAMENTO
    /* WHERE CD_CLI = 513011666 */
    ORDER BY TS_EXECUCAO
  );

  DISCONNECT FROM mypg;
QUIT;

PROC SQL;
  CREATE TABLE work.logs_dedup AS
  SELECT
      cd_cli,
      ts_aclt_slc,
      nr_seql_slct,
      nr_seql_pecs_nvl,
      status AS cod_status,
      MAX(ts_execucao) AS ts_execucao FORMAT=datetime26.6
  FROM work.logs_gerenciamento
  GROUP BY
      cd_cli,
      ts_aclt_slc,
      nr_seql_slct,
      nr_seql_pecs_nvl,
      status
  ;
QUIT;

PROC SQL;
  CREATE TABLE work.logs_dedup_map AS
  SELECT
      a.cd_cli,
      a.ts_aclt_slc,
      a.nr_seql_slct,
      a.nr_seql_pecs_nvl,
      b.nome_col,
      a.ts_execucao
  FROM work.logs_dedup a
  INNER JOIN work.map_status b
    ON a.cod_status = b.cod_status
  ;
QUIT;

PROC SORT DATA=work.logs_dedup_map;
  BY cd_cli ts_aclt_slc nr_seql_slct nr_seql_pecs_nvl;
RUN;

PROC TRANSPOSE DATA=work.logs_dedup_map
               OUT=work.logs_gerenciamento_pivot(DROP=_name_);
  BY cd_cli ts_aclt_slc nr_seql_slct nr_seql_pecs_nvl;
  ID nome_col;
  VAR ts_execucao;
RUN;

DATA work.logs_gerenciamento_final;
  SET work.logs_gerenciamento_pivot;

  /* ---- DROP: se houve cancelamento, remove a linha ---- */
  IF NOT MISSING(dt_cancelamento) THEN DELETE;

  if missing(dt_ini_analise) then dt_ini_analise = dt_entrada;

  FORMAT dt_devolucao dt_finalizacao datetime26.6;

  /* ---- Consolidação DEVOLUÇÃO (3 colunas -> 1) ---- */
  ARRAY devs[3] dt_devolucao dt_devolucao2 dt_devolucao3;
  dt_devolucao = .;

  DO i = 1 TO DIM(devs);
    IF NOT MISSING(devs[i]) THEN DO;
      IF MISSING(dt_devolucao) THEN dt_devolucao = devs[i];
      ELSE dt_devolucao = MIN(dt_devolucao, devs[i]);
    END;
  END;

  /* ---- Consolidação FINALIZAÇÃO (2 colunas -> 1) ---- */
  ARRAY fins[2] dt_finalizacao dt_finalizacao2;
  dt_finalizacao = .;

  DO j = 1 TO DIM(fins);
    IF NOT MISSING(fins[j]) THEN DO;
      IF MISSING(dt_finalizacao) THEN dt_finalizacao = fins[j];
      ELSE dt_finalizacao = MIN(dt_finalizacao, fins[j]);
    END;
  END;

  DROP i j dt_devolucao2 dt_devolucao3 dt_finalizacao2;
RUN;

PROC PRINT DATA=work.logs_gerenciamento_final(OBS=5);
RUN;

Obs,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl,dt_entrada,dt_ini_comite,dt_ini_comite_segundo_voto,dt_finalizacao,dt_saida,dt_ini_analise,dt_devolucao,dt_cancelamento
1,7023,25FEB2025:13:16:18.300712,1,2,26FEB2025:10:03:24.384840,26FEB2025:11:28:14.139799,27FEB2025:10:18:29.802828,.,27FEB2025:18:01:14.625138,26FEB2025:10:03:24.384840,.,.
2,7023,25FEB2025:13:16:18.300712,2,2,28MAR2025:14:51:14.965495,28MAR2025:16:24:05.942024,28MAR2025:18:03:38.042674,.,31MAR2025:09:01:06.918926,28MAR2025:16:00:10.111777,.,.
3,7069,05MAR2026:11:56:23.014712,1,2,05MAR2026:17:31:02.303430,09MAR2026:09:05:49.344306,09MAR2026:13:46:32.171109,.,10MAR2026:18:01:14.371697,09MAR2026:08:10:43.929212,.,.
4,7069,05MAR2026:11:56:23.014712,2,2,12MAR2026:08:49:28.766013,12MAR2026:10:30:35.459503,.,.,.,12MAR2026:09:03:12.012276,.,.
5,7195,01APR2025:11:12:16.015481,1,2,01APR2025:13:41:11.704751,02APR2025:11:16:28.075724,02APR2025:14:09:11.554174,.,02APR2025:17:30:54.796741,01APR2025:13:41:21.661869,.,.


In [4]:
/* === TABELA PARA CAPTURA DA MATRÍCULA DO RESPONSÁVEL PELA ANÁLISE === */

PROC SQL;
  RESET EXEC;

  CONNECT TO POSTGRES AS MYPG
    (server   = "172.16.14.11"
     port     = 5432
     database = "curadoria"
     user     = "painel_user"
     password = "painel_user"
     readbuff = 10000
     conopts  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE WORK.finalizados_matricula AS
  SELECT * FROM CONNECTION TO MYPG
  (
    SELECT
        cd_cli,
        ts_aclt_slc,
        nr_seql_slct,
        nr_seql_pecs_nvl,
        matricula_origem,
        ts_execucao
    FROM HIGHVAREJO.logs_gerenciamento
    WHERE status = 2
    ORDER BY ts_execucao
  );

  DISCONNECT FROM MYPG;
QUIT;

PROC PRINT DATA=WORK.finalizados_matricula (OBS=3);
RUN;

Obs,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl,matricula_origem,ts_execucao
1,514387611,18SEP2024:11:18:11.921003,1,2,F6828773,20SEP2024:13:34:18.898621
2,932759001,25SEP2024:10:40:42.172080,1,2,F9556837,25SEP2024:13:29:51.833322
3,514387611,24SEP2024:10:20:36.396716,1,2,F6828773,25SEP2024:13:33:20.697809


In [5]:
/* === TABELA INFORMAÇÕES FUNCIONÁRIOS === */

PROC SQL;
  RESET EXEC;

  CONNECT TO POSTGRES AS MYPG
    (server   = "172.16.14.11"
     port     = 5432
     database = "curadoria"
     user     = "painel_user"
     password = "painel_user"
     readbuff = 10000
     conopts  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE WORK.funcionarios AS
  SELECT * FROM CONNECTION TO MYPG
  (
    SELECT *
    FROM DMTV_GEREM.funcionarios
  );

  DISCONNECT FROM MYPG;
QUIT;

PROC PRINT DATA=WORK.funcionarios (OBS=3);
RUN;

Obs,chave,nome,nome_guerra,prefixo,cd_funcao,inativo,perfil
1,F0125841,ADEMILSON MEURER,ADEMILSON MEURER,8624,12330,0,1
2,F0145517,ADILSON BASILIO DE SOUZA,ADILSON,8624,12381,0,2
3,F0153141,ADMA APARECIDA ASSUNCAO DA CRUZ,ADMA,8624,12392,0,1


In [6]:
/* === TABELA INFORMAÇÕES AGENCIAS === */

libname mydb mysql
       server="10.2.97.159"
       port=3306
       database="painel_dicre"
       user="painel_dicre"
       password="painel"
       dbconinit="SET NAMES utf8mb4"
       PRESERVE_COL_NAMES=YES
       PRESERVE_TAB_NAMES=YES;

PROC SQL;
  CONNECT USING mydb;

  CREATE TABLE work.agencia AS
  SELECT *
  FROM CONNECTION TO mydb
  (
    SELECT *
    FROM tb_agencia
  );

  DISCONNECT FROM mydb;
QUIT;

PROC PRINT DATA=work.agencia (OBS=5);
RUN;

Obs,prefixo,nm_agencia,nm_super,nm_diretoria,estado,regiao,pilar,tipo_mercado,nm_regional,prf_super
1,0,,,,,,DEMAIS,D_AUDIT,,
2,0001,RIO,SUPER VAREJO RJ,DIVAR-COMERCIAL VAR,RJ,SUDESTE,VAREJO,PJ MIDDLE + UPPER + IMOB PJ,SUP ESP RJ CAP,8491
3,0002,MANAUS,SUPER VAREJO N I,DIVAR-COMERCIAL VAR,AM,NORTE,VAREJO,PJ MIDDLE + UPPER + IMOB PJ,SUP AM,8494
4,0003,PRESIDENTE VARGAS,SUPER VAREJO N II,DIVAR-COMERCIAL VAR,PA,NORTE,VAREJO,PJ MIDDLE + UPPER + IMOB PJ,SUP BELEM,8501
5,0004,SANTOS,SUPER VAR GRANDE SP,DIVAR-COMERCIAL VAR,SP,SUDESTE,VAREJO,PJ MIDDLE + UPPER + IMOB PJ,SUP SAO BERN.CAMPO,8508


In [7]:
/* ==== JOIN DAS TABELAS PARA GERAÇÃO DO DATAFRAME FINAL ==== */

%let dt_hoje = %sysfunc(today());

PROC SQL;
  CREATE TABLE work.painel_calendario AS
  SELECT DISTINCT
      p.id,
      p.protocolo,
      p.nome_cliente,
      PUT(p.cod_mci, BEST32.) AS cod_mci LENGTH=32 FORMAT=$32.,
      p.dta_entrada format=date9. as dta_entrada,
      p.dta_saida format=date9. as dta_saida,
      p.dta_ini_etapa format=date9. as dta_ini_etapa, /* p.dta_ini_etapa, */
      p.etapa,
      p.tp_finalizacao,
      p.tipo_processo,
      p.tipo_produto,
      p.gerencia,
      p.tipo_cliente,
      p.pilar,
      p.unidade_analise,
      PUT(p.prefixo_agencia, Z4.) AS prefixo_agencia LENGTH=4 FORMAT=$4.,
      p.funci_etapa,
      p.equipe,
      p.gerente,
      p.dta_vencimento_limite format=date9. as dta_vencimento_limite, /* p.dta_vencimento_limite, */
      p.vlr_limite_atual,
      p.vlr_limite_solicitado,
      p.dta_pacto format=date9. as dta_pacto, /* p.dta_pacto, */
      p.dta_estoque format=date9. as dta_estoque, /* p.dta_estoque, */

      /* COALESCE(
          (SELECT SUM(c_dm.flg_dia_util)
             FROM work.calendario c_dm
             WHERE c_dm.data_ref  >  p.dta_entrada
                 AND c_dm.data_ref <= &dt_hoje
              ),
              0
        ) AS dias_uteis_dm, Dias úteis entre ENTRADA e HOJE */
      
      COALESCE(
          (
            SELECT SUM(c_dm.flg_dia_util)
            FROM work.calendario c_dm
            WHERE c_dm.data_ref  >  p.dta_entrada
                AND c_dm.data_ref <= 
                  CASE 
                    WHEN p.dta_saida IS MISSING OR p.dta_saida = 0
                      THEN &dt_hoje
                    ELSE p.dta_saida
                  END
            ),
            0
        ) AS dias_uteis_dm,   /* Dias úteis entre ENTRADA e SAÍDA (ou HOJE se saída nula) */
      
      p.dias_corridos_dm,

      COALESCE(
          (SELECT SUM(c_pacto.flg_dia_util)
             FROM work.calendario c_pacto
             WHERE c_pacto.data_ref >  &dt_hoje
                 AND c_pacto.data_ref <= p.dta_pacto
              ),
              0
        ) AS dias_uteis_pacto, /*  Dias úteis entre HOJE e o pacto */
      
      p.dias_corridos_pacto,

      COALESCE(
          (SELECT SUM(c_etapa.flg_dia_util)
             FROM work.calendario c_etapa
             WHERE c_etapa.data_ref >  p.dta_ini_etapa
                 AND c_etapa.data_ref <= &dt_hoje
              ),
              0
        ) AS dias_uteis_etapa, /* Dias úteis entre o início da etapa e a data de hoje ########### REVISAR ################## */
      
      p.dias_corridos_etapa,

      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 0 AND 4
        THEN 1
        ELSE 0
      END AS "0_4d"n, /* dias_uteis_dm - 0_4d: flag 0 a 4 dias úteis = TRUE */

      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 5 AND 9
        THEN 1
        ELSE 0
      END AS "5_9d"n, /* 5_9d: flag 5 a 9 dias úteis = TRUE */
    
      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 10 AND 14
        THEN 1
        ELSE 0
      END AS "10_14d"n, /* 10_14d: flag 10 a 14 dias úteis = TRUE */
    
      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 15 AND 19
        THEN 1
        ELSE 0
      END AS "15_19d"n, /* 15_19d: flag 15 a 19 dias úteis = TRUE */

      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 20 AND 24
        THEN 1
        ELSE 0
      END AS "20_24d"n, /* 20_24d: flag 20 a 24 dias úteis = TRUE */
    
      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 25 AND 29
        THEN 1
        ELSE 0
      END AS "25_29d"n, /* 25_29d: flag 25 a 29 dias úteis = TRUE */

      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 30 AND 39
        THEN 1
        ELSE 0
      END AS "30_39d"n, /* 30_39d: flag 30 a 39 dias úteis = TRUE */

      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 40 AND 49
        THEN 1
        ELSE 0
      END AS "40_49d"n, /* 40_49d: flag 40 a 49 dias úteis = TRUE */

      CASE
        WHEN CALCULATED dias_uteis_dm BETWEEN 50 AND 59
        THEN 1
        ELSE 0
      END AS "50_59d"n, /* 50_59d: flag 50 a 59 dias úteis = TRUE */

      CASE
        WHEN CALCULATED dias_uteis_dm >= 60
        THEN 1
        ELSE 0
      END AS "60_d"n, /* 60_d: flag 60 dias úteis ou mais = TRUE */

      p.tipo_dd,
      p.dt_ini_triagem,
      p.dt_fim_triagem,
      
      p.dt_ini_ag_analise,
      p.dt_fim_ag_analise,
      
      DATEPART(lgf.dt_ini_analise) AS dt_ini_analise FORMAT=DATE9.,
      DATEPART(lgf.dt_ini_comite) AS dt_fim_analise FORMAT=DATE9.,
      p.dt_ini_diligencia,
      p.dt_fim_diligencia,
      
      DATEPART(lgf.dt_ini_comite) AS dt_ini_comite FORMAT=DATE9.,
      DATEPART(lgf.dt_saida) AS dt_fim_comite FORMAT=DATE9.,
      
      p.dt_ini_esc_superior,
      p.dt_fim_esc_superior,
      p.tm_grp,
      fun.nome AS funci_analise,
      age.nm_super,
      age.nm_regional,
      age.nm_agencia,
      '' AS marcador,
      p.matricula_funci_etapa,
      p.matricula_gerente,
      mat.matricula_origem AS matricula_funci_analise
      
      /* p.cd_cli, */
      /* p.ts_aclt_slc, */
      /* p.nr_seql_slct, */
      /* p.nr_seql_pecs_nvl */

  FROM work.inicial_painel AS p

  LEFT JOIN work.logs_gerenciamento_final AS lgf
    ON p.cd_cli           = lgf.cd_cli
   AND p.ts_aclt_slc      = lgf.ts_aclt_slc
   AND p.nr_seql_slct     = lgf.nr_seql_slct
   AND p.nr_seql_pecs_nvl = lgf.nr_seql_pecs_nvl

  LEFT JOIN work.finalizados_matricula AS mat
    ON p.cd_cli           = mat.cd_cli
   AND p.ts_aclt_slc      = mat.ts_aclt_slc
   AND p.nr_seql_slct     = mat.nr_seql_slct
   AND p.nr_seql_pecs_nvl = mat.nr_seql_pecs_nvl

  LEFT JOIN work.funcionarios fun
    ON fun.chave          = mat.matricula_origem
    
  LEFT JOIN work.agencia age
    ON cats(age.prefixo) = putn(p.prefixo_agencia, 'Z4.')
    /* ON age.prefixo        = PUT(p.prefixo_agencia, Z4.) */
      
  ;
  
QUIT;

PROC PRINT DATA=work.painel_calendario(OBS=3);
RUN;

Obs,id,protocolo,nome_cliente,cod_mci,dta_entrada,dta_saida,dta_ini_etapa,etapa,tp_finalizacao,tipo_processo,tipo_produto,gerencia,tipo_cliente,pilar,unidade_analise,prefixo_agencia,funci_etapa,equipe,gerente,dta_vencimento_limite,vlr_limite_atual,vlr_limite_solicitado,dta_pacto,dta_estoque,dias_uteis_dm,dias_corridos_dm,dias_uteis_pacto,dias_corridos_pacto,dias_uteis_etapa,dias_corridos_etapa,0_4d,5_9d,10_14d,15_19d,20_24d,25_29d,30_39d,40_49d,50_59d,60_d,tipo_dd,dt_ini_triagem,dt_fim_triagem,dt_ini_ag_analise,dt_fim_ag_analise,dt_ini_analise,dt_fim_analise,dt_ini_diligencia,dt_fim_diligencia,dt_ini_comite,dt_fim_comite,dt_ini_esc_superior,dt_fim_esc_superior,tm_grp,funci_analise,nm_super,nm_regional,nm_agencia,marcador,matricula_funci_etapa,matricula_gerente,matricula_funci_analise
1,0,202600280586,RURAL TECH COMERCIO E SERVICOS LTDA,445064998,04MAR2026,.,10MAR2026,COMITÊ,,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,GEREM,P. JURÍDICA,VAREJO,GEREM-SP ANALÍTICO,0452,PAULO LAVINIO DE ARAUJO JUNIOR,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,24JAN2027,1000000.00,,14MAR2026,12MAR2026,6,8,1,7,2,2,0,1,0,0,0,0,0,0,0,0,USO,.,.,.,.,06MAR2026,06MAR2026,.,.,06MAR2026,.,.,.,.,RAFAEL LUIZ MIRANDA,SUPER PJ II,SUP PJ C O I,HIGH EMPRESA BSB,,F8125991,F0281079,F8367923
2,0,202600282205,SUPERMERCADO VENEZA LTDA - EPP,380530,06MAR2026,.,10MAR2026,COMITÊ,,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,GEREM,P. JURÍDICA,VAREJO,GEREM-SP ANALÍTICO,0773,PAULO LAVINIO DE ARAUJO JUNIOR,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,18FEB2026,1000000.00,,16MAR2026,12MAR2026,4,6,2,9,2,2,1,0,0,0,0,0,0,0,0,0,USO,.,.,.,.,09MAR2026,09MAR2026,.,.,09MAR2026,.,.,.,.,LUCIANA NUNES CORDEIRO TUPYNAMBA,SUPER VAREJO PR,SUP MARINGA,MANDAGUACU,,F8125991,F0281079,F6323607
3,0,202600282206,PROPILEU SANEAMENTO E CONSTRUCOES LTDA.,32300523,06MAR2026,.,10MAR2026,COMITÊ,,RECONSID,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,GEREM,P. JURÍDICA,VAREJO,GEREM-SP ANALÍTICO,0386,PAULO LAVINIO DE ARAUJO JUNIOR,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,25FEB2027,3020000.00,,16MAR2026,12MAR2026,4,6,2,9,2,2,1,0,0,0,0,0,0,0,0,0,USO,.,.,.,.,09MAR2026,09MAR2026,.,.,09MAR2026,.,.,.,.,LUCIANA NUNES CORDEIRO TUPYNAMBA,SUPER PJ I,SUP PJ SP CAPITAL,EMPRESA SANTANA,,F8125991,F0281079,F6323607


In [8]:
LIBNAME HV POSTGRES
  SERVER   = "172.16.14.11"
  PORT     = 5432
  DATABASE = "curadoria"
  USER     = "painel_user"
  PASSWORD = "painel_user"
  SCHEMA   = "highvarejo"
  PRESERVE_COL_NAMES = YES
  PRESERVE_TAB_NAMES = YES
  READBUFF = 10000
  CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
;

/* remove a tabela no Postgres */
PROC SQL;
  CONNECT TO POSTGRES AS mypg
  ( SERVER="172.16.14.11" PORT=5432 DATABASE="curadoria"
    USER="painel_user" PASSWORD="painel_user"
    READBUFF=10000
    CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
  );

  EXECUTE (DROP TABLE IF EXISTS highvarejo.tb_estoque_principal_geral_62) BY mypg;

  DISCONNECT FROM mypg;
QUIT;

/* cria e carrega a tabela no Postgres */
DATA HV.tb_estoque_principal_geral_62;
  SET WORK.PAINEL_CALENDARIO;
RUN;

LIBNAME HV CLEAR;

19                                                         The SAS System                             12:49 Thursday, March 12, 2026

727        ods listing close;ods html5 (id=saspy_internal) file=_tomods1 options(bitmap_mode='inline') device=svg style=HTMLBlue;
727      ! ods graphics on / outputfmt=png;
NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1
728        
729        LIBNAME HV POSTGRES
730          SERVER   = "172.16.14.11"
731          PORT     = 5432
732          DATABASE = "curadoria"
733          USER     = "painel_user"
734          PASSWORD = XXXXXXXXXXXXX
735          SCHEMA   = "highvarejo"
736          PRESERVE_COL_NAMES = YES
737          PRESERVE_TAB_NAMES = YES
738          READBUFF = 10000
739          CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
740        ;
NOTE: Libref HV was successfully assigned as follows: 
      Engine:        POSTGRES 
      Physical Name: 172.16.14.11
741        
742        /* remove a tabela no Postgres */


In [9]:
/*==== NORMALIZAR TIPOS DAS DATAS ====*/

DATA WORK.PAINEL_CALENDARIO;
  SET WORK.PAINEL_CALENDARIO
    (RENAME=(
      dta_estoque = _dta_estoque
      dta_entrada = _dta_entrada
      dta_saida   = _dta_saida
      dta_pacto   = _dta_pacto
    ));

  LENGTH dta_estoque dta_entrada dta_saida dta_pacto 8;
  FORMAT dta_estoque dta_entrada dta_saida dta_pacto DATE9.;

  /* ---- dta_estoque ---- */
  IF VTYPE(_dta_estoque) = 'C' THEN DO;
    _tmp = INPUTN(_dta_estoque,'ANYDTDTE.');
    IF _tmp > 1000000 THEN dta_estoque = DATEPART(_tmp);
    ELSE dta_estoque = _tmp;
  END;
  ELSE DO;
    IF _dta_estoque > 1000000 THEN dta_estoque = DATEPART(_dta_estoque);
    ELSE dta_estoque = _dta_estoque;
  END;

  /* ---- dta_entrada ---- */
  IF VTYPE(_dta_entrada) = 'C' THEN DO;
    _tmp = INPUTN(_dta_entrada,'ANYDTDTE.');
    IF _tmp > 1000000 THEN dta_entrada = DATEPART(_tmp);
    ELSE dta_entrada = _tmp;
  END;
  ELSE DO;
    IF _dta_entrada > 1000000 THEN dta_entrada = DATEPART(_dta_entrada);
    ELSE dta_entrada = _dta_entrada;
  END;

  /* ---- dta_saida ---- */
  IF VTYPE(_dta_saida) = 'C' THEN DO;
    _tmp = INPUTN(_dta_saida,'ANYDTDTE.');
    IF _tmp > 1000000 THEN dta_saida = DATEPART(_tmp);
    ELSE dta_saida = _tmp;
  END;
  ELSE DO;
    IF _dta_saida > 1000000 THEN dta_saida = DATEPART(_dta_saida);
    ELSE dta_saida = _dta_saida;
  END;

  /* ---- dta_pacto ---- */
  IF VTYPE(_dta_pacto) = 'C' THEN DO;
    _tmp = INPUTN(_dta_pacto,'ANYDTDTE.');
    IF _tmp > 1000000 THEN dta_pacto = DATEPART(_tmp);
    ELSE dta_pacto = _tmp;
  END;
  ELSE DO;
    IF _dta_pacto > 1000000 THEN dta_pacto = DATEPART(_dta_pacto);
    ELSE dta_pacto = _dta_pacto;
  END;

  DROP _dta_estoque _dta_entrada _dta_saida _dta_pacto _tmp;
RUN;


/*==== CRIAR tb_estoque_resultado =====*/

PROC DATASETS LIB=WORK NOLIST;
  DELETE tb_estoque_resultado;
QUIT;

PROC SQL;
  CREATE TABLE tb_estoque_resultado AS
  SELECT
      dta_estoque,
      tipo_cliente,
      etapa,
      gerencia,
      unidade_analise,
      equipe,
      gerente,
      pilar,
      tipo_processo,
      nm_super,
      nm_regional,
      SUM(dias_uteis_dm)        AS dias_uteis_dm,
      SUM(dias_corridos_dm)     AS dias_corridos_dm,
      SUM(dias_uteis_pacto)     AS dias_uteis_pacto,
      SUM(dias_corridos_pacto)  AS dias_corridos_pacto,
      SUM(dias_uteis_etapa)     AS dias_uteis_etapa,
      SUM(dias_corridos_etapa)  AS dias_corridos_etapa,
      SUM('0_4d'n)              AS qt_00_04,
      SUM('5_9d'n)              AS qt_05_09,
      SUM('10_14d'n)            AS qt_10_14,
      SUM('15_19d'n)            AS qt_15_19,
      SUM('20_24d'n)            AS qt_20_24,
      SUM('25_29d'n)            AS qt_25_29,
      SUM('30_39d'n)            AS qt_30_39,
      SUM('40_49d'n)            AS qt_40_49,
      SUM('50_59d'n)            AS qt_50_59,
      SUM('60_d'n)              AS qt_m60,
      SUM(qt_entrada)           AS qt_entadas,
      SUM(qt_saida)             AS qt_saidas,
      SUM(qt_estoque)           AS qt_estoque,
      SUM(qt_registros)         AS qt_registros,
      SUM(qt_fora_pz)           AS qt_fora_pz,
      SUM(qt_antes_pz)          AS qt_antes_pz,
      SUM(qt_no_pz)             AS qt_no_pz,
      tipo_dd

  FROM
  (
      SELECT
          tipo_dd,
          dta_estoque,
          equipe,
          gerente,
          tipo_processo,
          pilar,
          unidade_analise,
          nm_super,
          nm_regional,
          tipo_cliente,
          gerencia,
          etapa,
          dias_uteis_dm,
          dias_corridos_dm,
          dias_uteis_pacto,
          dias_corridos_pacto,
          dias_uteis_etapa,
          dias_corridos_etapa,

          '0_4d'n,
          '5_9d'n,
          '10_14d'n,
          '15_19d'n,
          '20_24d'n,
          '25_29d'n,
          '30_39d'n,
          '40_49d'n,
          '50_59d'n,
          '60_d'n,

          CASE WHEN dta_entrada = dta_estoque THEN 1 ELSE 0 END AS qt_entrada,
          CASE WHEN dta_saida   = dta_estoque THEN 1 ELSE 0 END AS qt_saida,
          CASE WHEN dta_saida   = dta_estoque THEN 0 ELSE 1 END AS qt_estoque,
          1 AS qt_registros,

          CASE WHEN dta_pacto <  dta_estoque THEN 1 ELSE 0 END AS qt_fora_pz,
          CASE WHEN dta_pacto >  dta_estoque THEN 1 ELSE 0 END AS qt_antes_pz,
          CASE WHEN dta_pacto =  dta_estoque THEN 1 ELSE 0 END AS qt_no_pz

      FROM
          WORK.PAINEL_CALENDARIO
  ) v_cont_estoque

  GROUP BY
      dta_estoque,
      tipo_cliente,
      etapa,
      gerencia,
      unidade_analise,
      equipe,
      gerente,
      pilar,
      tipo_processo,
      nm_super,
      nm_regional,
      tipo_dd
  ;
QUIT;

PROC PRINT DATA=work.tb_estoque_resultado(OBS=3);
RUN;

Obs,dta_estoque,tipo_cliente,etapa,gerencia,unidade_analise,equipe,gerente,pilar,tipo_processo,nm_super,nm_regional,dias_uteis_dm,dias_corridos_dm,dias_uteis_pacto,dias_corridos_pacto,dias_uteis_etapa,dias_corridos_etapa,qt_00_04,qt_05_09,qt_10_14,qt_15_19,qt_20_24,qt_25_29,qt_30_39,qt_40_49,qt_50_59,qt_m60,qt_entadas,qt_saidas,qt_estoque,qt_registros,qt_fora_pz,qt_antes_pz,qt_no_pz,tipo_dd
1,12MAR2026,P. JURÍDICA,AGUARDANDO ANÁLISE,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,RECONSID,SUPER PJ I,SUP PJ RJ-ES,2,2,18,43,1545,0,3,0,0,0,0,0,0,0,0,0,1,0,3,3,0,3,0,USO
2,12MAR2026,P. JURÍDICA,AGUARDANDO ANÁLISE,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,RECONSID,SUPER PJ I,SUP PJ SC,2,2,12,28,1,1,2,0,0,0,0,0,0,0,0,0,0,0,2,2,0,2,0,USO
3,12MAR2026,P. JURÍDICA,AGUARDANDO ANÁLISE,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,RECONSID,SUPER PJ I,SUP PJ SP NOROESTE,1,1,6,14,1545,.,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,USO


In [10]:
LIBNAME HV POSTGRES
  SERVER   = "172.16.14.11"
  PORT     = 5432
  DATABASE = "curadoria"
  USER     = "painel_user"
  PASSWORD = "painel_user"
  SCHEMA   = "highvarejo"
  PRESERVE_COL_NAMES = YES
  PRESERVE_TAB_NAMES = YES
  READBUFF = 10000
  CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
;

/* remove a tabela no Postgres */
PROC SQL;
  CONNECT TO POSTGRES AS mypg
  ( SERVER="172.16.14.11" PORT=5432 DATABASE="curadoria"
    USER="painel_user" PASSWORD="painel_user"
    READBUFF=10000
    CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
  );

  EXECUTE (DROP TABLE IF EXISTS highvarejo.tb_estoque_resultado_62) BY mypg;

  DISCONNECT FROM mypg;
QUIT;

/* cria e carrega a tabela no Postgres */
DATA HV.tb_estoque_resultado_62;
  SET work.tb_estoque_resultado;
RUN;

LIBNAME HV CLEAR;

23                                                         The SAS System                             12:49 Thursday, March 12, 2026

949        ods listing close;ods html5 (id=saspy_internal) file=_tomods1 options(bitmap_mode='inline') device=svg style=HTMLBlue;
949      ! ods graphics on / outputfmt=png;
NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1
950        
951        LIBNAME HV POSTGRES
952          SERVER   = "172.16.14.11"
953          PORT     = 5432
954          DATABASE = "curadoria"
955          USER     = "painel_user"
956          PASSWORD = XXXXXXXXXXXXX
957          SCHEMA   = "highvarejo"
958          PRESERVE_COL_NAMES = YES
959          PRESERVE_TAB_NAMES = YES
960          READBUFF = 10000
961          CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
962        ;
NOTE: Libref HV was successfully assigned as follows: 
      Engine:        POSTGRES 
      Physical Name: 172.16.14.11
963        
964        /* remove a tabela no Postgres */


In [11]:
/* ===  ACESSO POSTGRESQL ESTOQUE METODOLOGIA 62 === */
/* === BLOCOS 1 A 4 === */


/* ========================= */
/* === BLOCO 1: STATUS (sa_f) === */
/* ========================= */
/* FILTRA SOMENTE STATUS FINALIZADOS */

PROC SQL;
  CONNECT TO POSTGRES AS mypg
    ( SERVER="172.16.14.11"
      PORT=5432
      DATABASE="curadoria"
      USER="painel_user"
      PASSWORD="painel_user"
      READBUFF=10000
      CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE work.sa_f AS
  SELECT *
  FROM CONNECTION TO mypg
  (
    SELECT
      cd_cli,
      ts_aclt_slc,
      nr_seql_slct,
      nr_seql_pecs_nvl,
      ts_execucao,
      status,
      nome_status
    FROM highvarejo.status_atual
    WHERE status IN (6, 98, 99)
  );

  DISCONNECT FROM mypg;
QUIT;


/* ============================ */
/* === BLOCO 2: PROTOCOLO (phv_f) === */
/* ============================ */

PROC SQL;
  CONNECT TO POSTGRES AS mypg
    ( SERVER="172.16.14.11"
      PORT=5432
      DATABASE="curadoria"
      USER="painel_user"
      PASSWORD="painel_user"
      READBUFF=10000
      CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE work.phv_f AS
  SELECT *
  FROM CONNECTION TO mypg
  (
    SELECT
      cd_cli,
      ts_aclt_slc,
      nr_seql_slct,
      nr_seql_pecs_nvl,
      ts_execucao,
      protocolo_ano,
      protocolo_num,
      entrada
    FROM highvarejo.protocolo
    WHERE numero IS NOT NULL
          /* AND cd_cli = 208112826 */ /* FILTRO DE TESTE (ORIGINAL) */
  );

  DISCONNECT FROM mypg;
QUIT;


/* ================================ */
/* === BLOCO 3: GERENCIAMENTO (gp) === */
/* ================================ */
/* BASE PRINCIPAL */

PROC SQL;
  CONNECT TO POSTGRES AS mypg
    ( SERVER="172.16.14.11"
      PORT=5432
      DATABASE="curadoria"
      USER="painel_user"
      PASSWORD="painel_user"
      READBUFF=10000
      CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE work.gp AS
  SELECT *
  FROM CONNECTION TO mypg
  (
    SELECT
      nom,
      cd_cli,
      ts_aclt_slc,
      nr_seql_slct,
      nr_seql_pecs_nvl,
      ts_execucao,
      tipo,
      pilar,
      atual
    FROM highvarejo.gerenciamento_painel
  );

  DISCONNECT FROM mypg;
QUIT;


/* ======================= */
/* === BLOCO 4: DDS (dds) === */
/* ======================= */

PROC SQL;
  CONNECT TO POSTGRES AS mypg
    ( SERVER="172.16.14.11"
      PORT=5432
      DATABASE="curadoria"
      USER="painel_user"
      PASSWORD="painel_user"
      READBUFF=10000
      CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
    );

  CREATE TABLE work.dds AS
  SELECT *
  FROM CONNECTION TO mypg
  (
    SELECT DISTINCT
      cd_cli,
      ts_aclt_slc,
      nr_seql_slct,
      nr_seql_pecs_nvl,
      ts_execucao,
      cd_depe_rsp_anl
    FROM public.dds_anl_hv
  );

  DISCONNECT FROM mypg;
QUIT;

25                                                         The SAS System                             12:49 Thursday, March 12, 2026

991        ods listing close;ods html5 (id=saspy_internal) file=_tomods1 options(bitmap_mode='inline') device=svg style=HTMLBlue;
991      ! ods graphics on / outputfmt=png;
NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1
992        
993        /* ===  ACESSO POSTGRESQL ESTOQUE METODOLOGIA 62 === */
994        /* === BLOCOS 1 A 4 === */
995        
996        
997        /* ========================= */
998        /* === BLOCO 1: STATUS (sa_f) === */
999        /* ========================= */
1000       /* FILTRA SOMENTE STATUS FINALIZADOS */
1001       
1002       PROC SQL;
1003         CONNECT TO POSTGRES AS mypg
1004           ( SERVER="172.16.14.11"
1005             PORT=5432
1006             DATABASE="curadoria"
1007             USER="painel_user"
1008             PASSWORD=XXXXXXXXXXXXX
1009             READBUFF=10000
1010             CONOPT

In [ ]:
/* === GERACAO DO ESTOQUE FINAL – METODOLOGIA 62 === */

PROC SQL;
  CREATE TABLE work.inicial_estoque AS
  SELECT
        0 AS id, /* id fixo sempre 0 */

        /* protocolo: ano + número com 8 dígitos */
        CATS(phv.protocolo_ano, PUT(phv.protocolo_num, Z8.)) AS protocolo,

        gp.nom AS nome_cliente,
        gp.cd_cli AS cod_mci,

        phv.entrada AS dta_entrada,
        DATEPART(sa.ts_execucao) AS dta_saida,

        phv.entrada + 10 AS dta_pacto, /* REUNIÃO COM LYGIA */

        UPCASE(gp.tipo) AS tipo_processo,
        'LIMITE DE CRÉDITO-PJ-METOD.ANALITICA' AS tipo_produto,
        'P. JURÍDICA' AS tipo_cliente,

        dds.cd_depe_rsp_anl AS prefixo_agencia, /* ###################### ################################# #################### REVISAR */
        UPCASE(gp.pilar) AS pilar,

        'GEREM-SP ANALÍTICO' AS unidade_analise, /* REUNIÃO COM LYGIA */
        'GEREM-SP DIV 5 PJ 62' AS equipe, /* ###################### ################################# #################### REVISAR */

        CASE
          WHEN gp.pilar = 'Atacado' THEN 'LYGIA CUNHA'
          ELSE 'ALBANO CARVALHO'
        END AS gerente,

        'GEREM' AS gerencia,

        CASE
          WHEN sa.status IS NULL THEN 'AGUARDANDO ANÁLISE'
          WHEN sa.status IN (-1, -2, 6, 98, 99) THEN 'FINALIZADO'
          WHEN sa.status = 1 THEN 'AGUARDANDO ANÁLISE'
          WHEN sa.status = 2 THEN 'EM ANÁLISE'
          WHEN sa.status IN (3, 4) THEN 'COMITÊ'
          WHEN sa.status IN (5, 95) THEN 'EM ENCERRAMENTO'
          ELSE sa.nome_status
        END AS etapa,

        CASE
          WHEN sa.status IN (-1, -2, 98) THEN 'DEVOLVIDO'
          WHEN sa.status = 6 THEN 'CONCLUÍDO'
          WHEN sa.status = 99 THEN 'CANCELADO'
          ELSE ''
        END AS tp_finalizacao,

        phv.entrada AS dta_registro,

        . AS dt_ini_diligencia,
        . AS dt_fim_diligencia,

        . AS dt_ini_ag_analise,
        . AS dt_fim_ag_analise,

        . AS dt_ini_triagem,
        . AS dt_fim_triagem,

        . AS dt_ini_esc_superior,
        . AS dt_fim_esc_superior,

        . AS dias_uteis_triagem,
        . AS dias_corridos_triagem,
        . AS dias_uteis_ag_analise,
        . AS dias_corridos_ag_analise,

        . AS dias_uteis_diligencia,
        . AS dias_corridos_diligencia,

        . AS dias_uteis_esc_superior,
        . AS dias_corridos_esc_superior,

        gp.atual AS vlr_limite_atual,

        '' AS vlr_limite_solicitado, /* ################################################ REVISAR */
        '' AS oper_estruturada, /* ################################### REVISAR */

        CASE
          WHEN gp.pilar = 'Atacado' THEN 'PJ MIDDLE + UPPER + IMOB PJ'
          ELSE 'VAREJO + MPE'
        END AS mercado, 

        'NÃO' AS processos_sla, /* REUNIÃO COM LYGIA */
        10 AS pacto_sla, /* REUNIÃO COM LYGIA */

        phv.protocolo_ano,
        phv.protocolo_num,

        gp.cd_cli,
        gp.ts_aclt_slc,
        gp.nr_seql_slct,
        gp.nr_seql_pecs_nvl

  FROM work.gp gp
  INNER JOIN work.sa_f sa
    ON sa.cd_cli = gp.cd_cli
   AND sa.ts_aclt_slc = gp.ts_aclt_slc
   AND sa.nr_seql_slct = gp.nr_seql_slct
   AND sa.nr_seql_pecs_nvl = gp.nr_seql_pecs_nvl

  INNER JOIN work.phv_f phv
    ON phv.cd_cli = gp.cd_cli
   AND phv.ts_aclt_slc = gp.ts_aclt_slc
   AND phv.nr_seql_slct = gp.nr_seql_slct
   AND phv.nr_seql_pecs_nvl = gp.nr_seql_pecs_nvl
   AND phv.ts_execucao = gp.ts_execucao

  LEFT JOIN work.dds dds
    ON dds.cd_cli = gp.cd_cli
   AND dds.ts_aclt_slc = gp.ts_aclt_slc
   AND dds.nr_seql_slct = gp.nr_seql_slct
   AND dds.nr_seql_pecs_nvl = gp.nr_seql_pecs_nvl
   AND dds.ts_execucao = gp.ts_execucao
 
 WHERE dds.cd_depe_rsp_anl IS NOT MISSING /* ###################### ################################# #################### REVISAR */
   
   ;
QUIT;

PROC PRINT DATA=work.inicial_estoque(OBS=3);
RUN;

Obs,id,protocolo,nome_cliente,cod_mci,dta_entrada,dta_saida,dta_pacto,tipo_processo,tipo_produto,tipo_cliente,prefixo_agencia,pilar,unidade_analise,equipe,gerente,gerencia,etapa,tp_finalizacao,dta_registro,dt_ini_diligencia,dt_fim_diligencia,dt_ini_ag_analise,dt_fim_ag_analise,dt_ini_triagem,dt_fim_triagem,dt_ini_esc_superior,dt_fim_esc_superior,dias_uteis_triagem,dias_corridos_triagem,dias_uteis_ag_analise,dias_corridos_ag_analise,dias_uteis_diligencia,dias_corridos_diligencia,dias_uteis_esc_superior,dias_corridos_esc_superior,vlr_limite_atual,vlr_limite_solicitado,oper_estruturada,mercado,processos_sla,pacto_sla,protocolo_ano,protocolo_num,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl
1,0,202500006868,INDUSTRIA E COMERCIO HIDROMAR LTDA,7023,25FEB2025,23799,23807,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2755,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,25FEB2025,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,3500000.00,,,VAREJO + MPE,NÃO,10,2025,6868,7023,25FEB2025:13:16:18.300712,1,2
2,0,202500011499,INDUSTRIA E COMERCIO HIDROMAR LTDA,7023,28MAR2025,23831,23838,RECONSID,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2755,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,28MAR2025,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,3500000.00,,,VAREJO + MPE,NÃO,10,2025,11499,7023,25FEB2025:13:16:18.300712,2,2
3,0,202600282204,REBOUCAS INDUSTRIA DE PLASTICOS LTDA,7069,05MAR2026,24175,24180,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2755,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,05MAR2026,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,.,9000000.00,,,VAREJO + MPE,NÃO,10,2026,282204,7069,05MAR2026:11:56:23.014712,1,2


In [ ]:
/* ===========================================================
   AJUSTES DE LOG (opcional, mas útil para enxergar gargalo)
   - MSGLEVEL=I: mostra no log se índices foram usados
   - FULLSTIMER: detalha tempo/CPU/IO por etapa
   =========================================================== */
OPTIONS MSGLEVEL=I FULLSTIMER; /* [2](SAS Certification Prep Guide Advanced Programming for SAS 9, Fourth Edition.pdf */

/* ===========================================================
   DATA DE HOJE (SAS DATE)
   =========================================================== */
%LET dt_hoje = %SYSFUNC(TODAY());

/* ===========================================================
   0) CALENDÁRIO COM ACUMULADO DE DIAS ÚTEIS (CUM_UTEIS)
   IDEIA:
     cum_uteis(d) = soma de flg_dia_util até a data d (inclusive)
   ENTÃO:
     dias_uteis ENTRE ( > dt_ini ) E ( <= dt_fim )
     = cum_uteis(dt_fim) - cum_uteis(dt_ini)
   =========================================================== */

/* Garante ordenação por data (fundamental para acumulado) */
PROC SORT DATA=work.calendario OUT=work.calendario_ord;
  BY data_ref;
RUN;

DATA work.calendario_cum;
  SET work.calendario_ord;
  BY data_ref;
  RETAIN cum_uteis 0;
  /* flg_dia_util deve ser 0/1. Se vier missing, trata como 0 */
  cum_uteis + COALESCE(flg_dia_util,0);
RUN;

/* Índice simples para acelerar joins por data_ref */
PROC DATASETS LIB=work NOLIST;
  MODIFY calendario_cum;
  INDEX CREATE data_ref;
QUIT;

/* ===========================================================
   1) LGF: PEGAR O ÚLTIMO REGISTRO POR CHAVE (DT_SAIDA MÁXIMA)
   Observação técnica:
     Seu subselect original com "SELECT lgf.* ... GROUP BY ... HAVING ..."
     pode ficar caro e ainda pode trazer linha não-determinística para colunas
     fora do GROUP BY. Aqui fazemos em 2 passos, determinístico e legível.
   =========================================================== */

PROC SQL;
  CREATE TABLE work.lgf_dt_saida_max AS
  SELECT
        lgf.cd_cli,
        lgf.ts_aclt_slc,
        lgf.nr_seql_slct,
        lgf.nr_seql_pecs_nvl,
        MAX(DATEPART(lgf.dt_saida)) AS dt_saida_max FORMAT=DATE9.
  FROM work.logs_gerenciamento_final AS lgf
  GROUP BY
        lgf.cd_cli,
        lgf.ts_aclt_slc,
        lgf.nr_seql_slct,
        lgf.nr_seql_pecs_nvl
  ;
QUIT;

PROC SQL;
  CREATE TABLE work.lgf_ult_raw AS
  SELECT lgf.*
  FROM work.logs_gerenciamento_final AS lgf
  INNER JOIN work.lgf_dt_saida_max AS m
    ON  lgf.cd_cli           = m.cd_cli
    AND lgf.ts_aclt_slc      = m.ts_aclt_slc
    AND lgf.nr_seql_slct     = m.nr_seql_slct
    AND lgf.nr_seql_pecs_nvl = m.nr_seql_pecs_nvl
    AND DATEPART(lgf.dt_saida) = m.dt_saida_max
  ;
QUIT;

/* Normaliza datas do LGF uma vez (evita repetir DATEPART no SELECT final) */
DATA work.lgf_ult;
  SET work.lgf_ult_raw;

  FORMAT
    dt_entrada_d      dt_saida_d      dt_saida_eff_d
    dt_ini_analise_d  dt_ini_comite_d  dt_ini_comite2_d
    dt_fim_analise_d  dt_fim_comite_d  DATE9.
  ;

  dt_entrada_d     = DATEPART(dt_entrada);
  dt_saida_d       = DATEPART(dt_saida);

  /* dt_saida_eff_d: se saída missing/0 -> hoje */
  IF MISSING(dt_saida_d) OR dt_saida_d = 0 THEN dt_saida_eff_d = &dt_hoje;
  ELSE dt_saida_eff_d = dt_saida_d;

  /* Mantém as "datas explícitas no select" já normalizadas */
  dt_ini_analise_d = DATEPART(dt_ini_analise);
  dt_fim_analise_d = DATEPART(dt_ini_comite);  /* seu dt_fim_analise = dt_ini_comite */

  dt_ini_comite_d  = DATEPART(dt_ini_comite);
  dt_fim_comite_d  = DATEPART(dt_finalizacao); /* dt_finalizacao já pode ser date/datetime; se for datetime, ajuste aqui */

  /* Caso dt_finalizacao seja DATETIME e você precise DATE:
     dt_fim_comite_d = DATEPART(dt_finalizacao);
     (deixe como está se dt_finalizacao já é DATE)
  */
RUN;

/* Índice nas chaves do LGF "último" para acelerar join final */
PROC DATASETS LIB=work NOLIST;
  MODIFY lgf_ult;
  INDEX CREATE cd_cli ts_aclt_slc nr_seql_slct nr_seql_pecs_nvl;
QUIT;

/* ===========================================================
   2) MATRÍCULA: PEGAR A MAIOR MATRICULA_ORIGEM POR CHAVE
   =========================================================== */

PROC SQL;
  CREATE TABLE work.mat_matricula_max AS
  SELECT
        mat.cd_cli,
        mat.ts_aclt_slc,
        mat.nr_seql_slct,
        mat.nr_seql_pecs_nvl,
        MAX(mat.matricula_origem) AS matricula_origem
  FROM work.finalizados_matricula AS mat
  GROUP BY
        mat.cd_cli,
        mat.ts_aclt_slc,
        mat.nr_seql_slct,
        mat.nr_seql_pecs_nvl
  ;
QUIT;

PROC DATASETS LIB=work NOLIST;
  MODIFY mat_matricula_max;
  INDEX CREATE cd_cli ts_aclt_slc nr_seql_slct nr_seql_pecs_nvl;
QUIT;

/* ===========================================================
   3) HISTÓRICO PRINCIPAL (AGORA SEM SUBQUERIES NO CALENDÁRIO)
   - Cada dias_uteis_* vira diferença de cum_uteis (2 joins por medida)
   - Mantém seus nomes e comentários
   =========================================================== */

PROC SQL;
  CREATE TABLE work.historico_principal AS
  SELECT
        p.id,

        /* =======================================================
           CHAVES (incluídas conforme solicitado)
           ======================================================= */
        p.cd_cli,
        p.ts_aclt_slc,
        p.nr_seql_slct,
        p.nr_seql_pecs_nvl,

        p.protocolo,
        p.nome_cliente,
        p.cod_mci,
        p.dta_entrada FORMAT=DATE9.,
        p.dta_saida   FORMAT=DATE9.,
        p.dta_pacto   FORMAT=DATE9.,
        p.tipo_processo,
        p.tipo_produto,
        p.tipo_cliente,
        p.prefixo_agencia,
        p.pilar,
        p.unidade_analise,
        p.equipe,
        p.gerente,
        p.gerencia,
        p.etapa,
        p.tp_finalizacao,
        p.dta_registro,

        /* =======================================================
           DATAS DO LGF (EXPLÍCITAS NO SELECT)
           (normalizadas em work.lgf_ult)
           ======================================================= */
        lgf.dt_ini_analise_d AS dt_ini_analise FORMAT=DATE9.,
        lgf.dt_fim_analise_d AS dt_fim_analise FORMAT=DATE9.,

        p.dt_ini_ag_analise FORMAT=DATE9.,
        p.dt_fim_ag_analise FORMAT=DATE9.,
        p.dt_ini_diligencia FORMAT=DATE9.,
        p.dt_fim_diligencia FORMAT=DATE9.,
        p.dt_ini_triagem FORMAT=DATE9.,
        p.dt_fim_triagem FORMAT=DATE9.,

        lgf.dt_ini_comite_d  AS dt_ini_comite FORMAT=DATE9.,
        lgf.dt_fim_comite_d  AS dt_fim_comite FORMAT=DATE9.,

        p.dt_ini_esc_superior FORMAT=DATE9.,
        p.dt_fim_esc_superior FORMAT=DATE9.,

        /* =======================================================
           1) DIAS ÚTEIS DM
           ENTRE:
             > DATEPART(lgf.dt_entrada)
             <= (DATEPART(lgf.dt_saida) OU HOJE)
           Fórmula via acumulado:
             cum(fim) - cum(inicio)
           ======================================================= */
        (COALESCE(c_dm_fim.cum_uteis,0) - COALESCE(c_dm_ini.cum_uteis,0)) AS dias_uteis_dm,

        /* =======================================================
           2) DIAS CORRIDOS DM
           (DATEPART(lgf.dt_saida) OU HOJE) - DATEPART(lgf.dt_entrada)
           ======================================================= */
        (lgf.dt_saida_eff_d - lgf.dt_entrada_d) AS dias_corridos_dm,

        p.dias_uteis_triagem,
        p.dias_corridos_triagem,
        p.dias_uteis_ag_analise,
        p.dias_corridos_ag_analise,

        /* =======================================================
           3) DIAS ÚTEIS ANÁLISE
           ENTRE:
             > p.dta_entrada
             <= (p.dta_saida OU HOJE)
           REGRAS:
             - SE DATA INÍCIO OU FIM = 0 OU NULL -> 0
             - SE RESULTADO < 0                -> 0
           ======================================================= */
        CASE
          /* Se alguma das datas não existir */
          WHEN p.dta_entrada IS NULL
            OR p.dta_entrada = 0
            OR (CASE
                  WHEN MISSING(p.dta_saida) OR p.dta_saida = 0
                    THEN &dt_hoje
                  ELSE p.dta_saida
                END) IS NULL
            OR (CASE
                  WHEN MISSING(p.dta_saida) OR p.dta_saida = 0
                    THEN &dt_hoje
                  ELSE p.dta_saida
                END) = 0
            THEN 0

          /* Se a subtração der negativa */
          WHEN (COALESCE(c_et_fim.cum_uteis,0) - COALESCE(c_et_ini.cum_uteis,0)) < 0
            THEN 0

          /* Caso válido */
          ELSE (COALESCE(c_et_fim.cum_uteis,0) - COALESCE(c_et_ini.cum_uteis,0))
        END AS dias_uteis_analise,

        /* =======================================================
           4) DIAS CORRIDOS ANÁLISE
           ======================================================= */
        (lgf.dt_saida_eff_d - lgf.dt_ini_analise_d) AS dias_corridos_analise,

        p.dias_uteis_diligencia,
        p.dias_corridos_diligencia,

        /* =======================================================
           5) DIAS ÚTEIS COMITÊ
           ENTRE:
             > DATEPART(lgf.dt_ini_comite)
             <= (DATEPART(lgf.dt_saida) OU HOJE)
           REGRAS:
             - datas nulas ou 0 => 0
             - resultado negativo => 0
           ======================================================= */
        CASE
          /* Se alguma das datas não existir */
          WHEN lgf.dt_ini_comite IS NULL
            OR lgf.dt_ini_comite = 0
            OR lgf.dt_saida_eff_d IS NULL
            OR lgf.dt_saida_eff_d = 0
          THEN 0

          /* Se o cálculo der negativo */
          WHEN (COALESCE(c_co_fim.cum_uteis,0) - COALESCE(c_co_ini.cum_uteis,0)) < 0
          THEN 0

          /* Caso normal */
          ELSE (COALESCE(c_co_fim.cum_uteis,0) - COALESCE(c_co_ini.cum_uteis,0))
        END AS dias_uteis_comite,

        /* =======================================================
           6) DIAS CORRIDOS COMITÊ
           ======================================================= */
        (lgf.dt_saida_eff_d - lgf.dt_ini_comite_d) AS dias_corridos_comite,

        p.dias_uteis_esc_superior,
        p.dias_corridos_esc_superior,

        p.vlr_limite_atual,
        p.vlr_limite_solicitado,
        p.oper_estruturada,
        p.mercado,
        p.processos_sla,
        p.pacto_sla,

        /* =======================================================
           7) DIAS ÚTEIS LÍQUIDO
           (igual ao seu modelo: dias_uteis_analise)
           ======================================================= */
        CALCULATED dias_uteis_analise AS dias_uteis_liquido,

        fun.nome AS funci_analise,
        age.nm_super,
        age.nm_regional,
        age.nm_agencia,

        mat.matricula_origem AS matricula_funci_analise,

        /* =======================================================
           8) DIAS ÚTEIS PACTO
           ENTRE:
             > (p.dta_pacto - 15)
             <= p.dta_pacto
           ======================================================= */
        (COALESCE(c_pa_fim.cum_uteis,0) - COALESCE(c_pa_ini.cum_uteis,0)) AS dias_uteis_pacto,

        /* =======================================================
           9) DIAS ÚTEIS ETAPA
           ENTRE:
             > p.dta_entrada
             <= (p.dta_saida OU HOJE)
           ======================================================= */
        (COALESCE(c_et_fim.cum_uteis,0) - COALESCE(c_et_ini.cum_uteis,0)) AS dias_uteis_etapa,

        /* =======================================================
           10) DIAS CORRIDOS ETAPA
           ======================================================= */
        (
          (CASE WHEN MISSING(p.dta_saida) OR p.dta_saida = 0 THEN &dt_hoje ELSE p.dta_saida END)
          - p.dta_entrada
        ) AS dias_corridos_etapa,

        /* =======================================================
           FLAGS DE FAIXA 
        ======================================================= */
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 0  AND 4  THEN 1 ELSE 0 END AS "0_4d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 5  AND 9  THEN 1 ELSE 0 END AS "5_9d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 10 AND 14 THEN 1 ELSE 0 END AS "10_14d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 15 AND 19 THEN 1 ELSE 0 END AS "15_19d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 20 AND 24 THEN 1 ELSE 0 END AS "20_24d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 25 AND 29 THEN 1 ELSE 0 END AS "25_29d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 30 AND 39 THEN 1 ELSE 0 END AS "30_39d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 40 AND 49 THEN 1 ELSE 0 END AS "40_49d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 50 AND 59 THEN 1 ELSE 0 END AS "50_59d"n,
        CASE WHEN CALCULATED dias_uteis_dm >= 60            THEN 1 ELSE 0 END AS "60_d"n,

        p.protocolo_ano,
        p.protocolo_num

  FROM work.inicial_estoque AS p

  /* LGF já “último por chave” */
  LEFT JOIN work.lgf_ult AS lgf
    ON  p.cd_cli           = lgf.cd_cli
    AND p.ts_aclt_slc      = lgf.ts_aclt_slc
    AND p.nr_seql_slct     = lgf.nr_seql_slct
    AND p.nr_seql_pecs_nvl = lgf.nr_seql_pecs_nvl

  /* Matrícula já “máxima por chave” */
  LEFT JOIN work.mat_matricula_max AS mat
    ON  p.cd_cli           = mat.cd_cli
    AND p.ts_aclt_slc      = mat.ts_aclt_slc
    AND p.nr_seql_slct     = mat.nr_seql_slct
    AND p.nr_seql_pecs_nvl = mat.nr_seql_pecs_nvl

  LEFT JOIN work.funcionarios AS fun
    ON fun.chave = mat.matricula_origem

  LEFT JOIN work.agencia AS age
    ON STRIP(age.prefixo) = PUT(p.prefixo_agencia, Z4.)

  /* =======================================================
     JOINS DO CALENDÁRIO (ACUMULADO) - 2 por métrica
     ======================================================= */

  /* DM: início = dt_entrada_d, fim = dt_saida_eff_d */
  LEFT JOIN work.calendario_cum AS c_dm_ini
    ON c_dm_ini.data_ref = lgf.dt_entrada_d
  LEFT JOIN work.calendario_cum AS c_dm_fim
    ON c_dm_fim.data_ref = lgf.dt_saida_eff_d

  /* ANÁLISE: início = dt_ini_analise_d, fim = dt_saida_eff_d */
  LEFT JOIN work.calendario_cum AS c_an_ini
    ON c_an_ini.data_ref = lgf.dt_ini_analise_d
  LEFT JOIN work.calendario_cum AS c_an_fim
    ON c_an_fim.data_ref = lgf.dt_saida_eff_d

  /* COMITÊ: início = dt_ini_comite_d, fim = dt_saida_eff_d */
  LEFT JOIN work.calendario_cum AS c_co_ini
    ON c_co_ini.data_ref = lgf.dt_ini_comite_d
  LEFT JOIN work.calendario_cum AS c_co_fim
    ON c_co_fim.data_ref = lgf.dt_saida_eff_d

  /* PACTO: início = (p.dta_pacto - 15), fim = p.dta_pacto */
  LEFT JOIN work.calendario_cum AS c_pa_ini
    ON c_pa_ini.data_ref = (p.dta_pacto - 15)
  LEFT JOIN work.calendario_cum AS c_pa_fim
    ON c_pa_fim.data_ref = p.dta_pacto

  /* ETAPA: início = p.dta_entrada, fim = (p.dta_saida OU HOJE) */
  LEFT JOIN work.calendario_cum AS c_et_ini
    ON c_et_ini.data_ref = p.dta_entrada
  LEFT JOIN work.calendario_cum AS c_et_fim
    ON c_et_fim.data_ref =
      (CASE WHEN MISSING(p.dta_saida) OR p.dta_saida = 0 THEN &dt_hoje ELSE p.dta_saida END)

  WHERE lgf.dt_ini_analise_d IS NOT NULL
  ;
QUIT;

PROC PRINT DATA=work.historico_principal(OBS=3);
RUN;

Obs,id,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl,protocolo,nome_cliente,cod_mci,dta_entrada,dta_saida,dta_pacto,tipo_processo,tipo_produto,tipo_cliente,prefixo_agencia,pilar,unidade_analise,equipe,gerente,gerencia,etapa,tp_finalizacao,dta_registro,dt_ini_analise,dt_fim_analise,dt_ini_ag_analise,dt_fim_ag_analise,dt_ini_diligencia,dt_fim_diligencia,dt_ini_triagem,dt_fim_triagem,dt_ini_comite,dt_fim_comite,dt_ini_esc_superior,dt_fim_esc_superior,dias_uteis_dm,dias_corridos_dm,dias_uteis_triagem,dias_corridos_triagem,dias_uteis_ag_analise,dias_corridos_ag_analise,dias_uteis_analise,dias_corridos_analise,dias_uteis_diligencia,dias_corridos_diligencia,dias_uteis_comite,dias_corridos_comite,dias_uteis_esc_superior,dias_corridos_esc_superior,vlr_limite_atual,vlr_limite_solicitado,oper_estruturada,mercado,processos_sla,pacto_sla,dias_uteis_liquido,funci_analise,nm_super,nm_regional,nm_agencia,matricula_funci_analise,dias_uteis_pacto,dias_uteis_etapa,dias_corridos_etapa,0_4d,5_9d,10_14d,15_19d,20_24d,25_29d,30_39d,40_49d,50_59d,60_d,protocolo_ano,protocolo_num
1,0,455152632,11FEB2025:16:55:00.930075,1,2,202500006260,FI MINERACAO LTDA,455152632,14FEB2025,17FEB2025,24FEB2025,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2578,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,14FEB2025,14FEB2025,14FEB2025,.,.,.,.,.,.,14FEB2025,.,.,.,1,3,.,.,.,.,1,3,.,.,1,3,.,.,0.00,,,VAREJO + MPE,NÃO,10,1,,SUPER VAR GRANDE SP,SUP SAO JOSE CAMPOS,SANTA ISABEL,,11,1,3,1,0,0,0,0,0,0,0,0,0,2025,6260
2,0,836879229,14FEB2025:11:07:03.696324,1,2,202500006273,ECCOMANIA COMERCIO DE COMBUSTIVEIS E DERIVADOS DE PETROLEO L,836879229,14FEB2025,17FEB2025,24FEB2025,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,46,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,14FEB2025,14FEB2025,14FEB2025,.,.,.,.,.,.,14FEB2025,.,.,.,1,3,.,.,.,.,1,3,.,.,1,3,.,.,3500000.00,,,VAREJO + MPE,NÃO,10,1,,SUPER PJ II,SUP PJ C O II,EMPRESA CUIABA,,11,1,3,1,0,0,0,0,0,0,0,0,0,2025,6273
3,0,447694844,14FEB2025:11:03:36.518236,1,2,202500006259,MADEMILE DERIVADOS DE MADEIRA LTDA,447694844,14FEB2025,17FEB2025,24FEB2025,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2389,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,14FEB2025,14FEB2025,17FEB2025,.,.,.,.,.,.,17FEB2025,.,.,.,1,3,.,.,.,.,1,3,.,.,0,0,.,.,4500000.00,,,VAREJO + MPE,NÃO,10,1,,SUPER VAREJO SC,SUP BLUMENAU,PAPANDUVA,,11,1,3,1,0,0,0,0,0,0,0,0,0,2025,6259


/* ===========================================================
   DATA DE HOJE (SAS DATE)
   =========================================================== */
%LET dt_hoje = %SYSFUNC(TODAY());

/* ===========================================================
   HISTÓRICO PRINCIPAL
   =========================================================== */
PROC SQL;
  CREATE TABLE work.historico_principal AS
  SELECT
        p.id,
        p.protocolo,
        p.nome_cliente,
        p.cod_mci,
        p.dta_entrada FORMAT=DATE9.,
        p.dta_saida   FORMAT=DATE9.,
        p.dta_pacto   FORMAT=DATE9.,
        p.tipo_processo,
        p.tipo_produto,
        p.tipo_cliente,
        p.prefixo_agencia,
        p.pilar,
        p.unidade_analise,
        p.equipe,
        p.gerente,
        p.gerencia,
        p.etapa,
        p.tp_finalizacao,
        p.dta_registro,

        /* =======================================================
           DATAS DO LGF (EXPLÍCITAS NO SELECT)
           ======================================================= */
        DATEPART(lgf.dt_ini_analise) AS dt_ini_analise FORMAT=DATE9.,
        DATEPART(lgf.dt_ini_comite)  AS dt_fim_analise FORMAT=DATE9.,

        p.dt_ini_ag_analise FORMAT=DATE9.,
        p.dt_fim_ag_analise FORMAT=DATE9.,
        p.dt_ini_diligencia FORMAT=DATE9.,
        p.dt_fim_diligencia FORMAT=DATE9.,
        p.dt_ini_triagem FORMAT=DATE9.,
        p.dt_fim_triagem FORMAT=DATE9.,

        DATEPART(lgf.dt_ini_comite)  AS dt_ini_comite FORMAT=DATE9.,
        lgf.dt_finalizacao           AS dt_fim_comite FORMAT=DATE9.,

        p.dt_ini_esc_superior FORMAT=DATE9.,
        p.dt_fim_esc_superior FORMAT=DATE9.,

        /* =======================================================
           1) DIAS ÚTEIS DM
           ENTRE:
             > DATEPART(lgf.dt_entrada)
             <= (DATEPART(lgf.dt_saida) OU HOJE)
           ======================================================= */
        (
          SELECT SUM(c.flg_dia_util)
          FROM work.calendario AS c
          WHERE c.data_ref > DATEPART(lgf.dt_entrada)
            AND c.data_ref <=
              CASE
                WHEN DATEPART(lgf.dt_saida) IS NULL
                  OR DATEPART(lgf.dt_saida) = 0
                THEN &dt_hoje
                ELSE DATEPART(lgf.dt_saida)
              END
        ) AS dias_uteis_dm,

        /* =======================================================
           2) DIAS CORRIDOS DM
           (DATEPART(lgf.dt_saida) OU HOJE) - DATEPART(lgf.dt_entrada)
           ======================================================= */
        (
          CASE
            WHEN DATEPART(lgf.dt_saida) IS NULL
              OR DATEPART(lgf.dt_saida) = 0
            THEN &dt_hoje
            ELSE DATEPART(lgf.dt_saida)
          END
          -
          DATEPART(lgf.dt_entrada)
        ) AS dias_corridos_dm,

        p.dias_uteis_triagem,
        p.dias_corridos_triagem,
        p.dias_uteis_ag_analise,
        p.dias_corridos_ag_analise,

        /* =======================================================
           3) DIAS ÚTEIS ANÁLISE
           ENTRE:
             > DATEPART(lgf.dt_ini_analise)
             <= (DATEPART(lgf.dt_saida) OU HOJE)
           ======================================================= */
        (
          SELECT SUM(c.flg_dia_util)
          FROM work.calendario AS c
          WHERE c.data_ref > DATEPART(lgf.dt_ini_analise)
            AND c.data_ref <=
              CASE
                WHEN DATEPART(lgf.dt_saida) IS NULL
                  OR DATEPART(lgf.dt_saida) = 0
                THEN &dt_hoje
                ELSE DATEPART(lgf.dt_saida)
              END
        ) AS dias_uteis_analise,

        /* =======================================================
           4) DIAS CORRIDOS ANÁLISE
           ======================================================= */
        (
          CASE
            WHEN DATEPART(lgf.dt_saida) IS NULL
              OR DATEPART(lgf.dt_saida) = 0
            THEN &dt_hoje
            ELSE DATEPART(lgf.dt_saida)
          END
          -
          DATEPART(lgf.dt_ini_analise)
        ) AS dias_corridos_analise,


        p.dias_uteis_diligencia,
        p.dias_corridos_diligencia,


        /* =======================================================
           5) DIAS ÚTEIS COMITÊ
           ENTRE:
             > DATEPART(lgf.dt_ini_comite)
             <= (DATEPART(lgf.dt_saida) OU HOJE)
           ======================================================= */
        (
          SELECT SUM(c.flg_dia_util)
          FROM work.calendario AS c
          WHERE c.data_ref > DATEPART(lgf.dt_ini_comite)
            AND c.data_ref <=
              CASE
                WHEN DATEPART(lgf.dt_saida) IS NULL
                  OR DATEPART(lgf.dt_saida) = 0
                THEN &dt_hoje
                ELSE DATEPART(lgf.dt_saida)
              END
        ) AS dias_uteis_comite,

        /* =======================================================
           6) DIAS CORRIDOS COMITÊ
           ======================================================= */
        (
          CASE
            WHEN DATEPART(lgf.dt_saida) IS NULL
              OR DATEPART(lgf.dt_saida) = 0
            THEN &dt_hoje
            ELSE DATEPART(lgf.dt_saida)
          END
          -
          DATEPART(lgf.dt_ini_comite)
        ) AS dias_corridos_comite,


        p.dias_uteis_esc_superior,
        p.dias_corridos_esc_superior,

        p.vlr_limite_atual,
        p.vlr_limite_solicitado,
        p.oper_estruturada,
        p.mercado,
        p.processos_sla,
        p.pacto_sla,

        /* =======================================================
           7) DIAS ÚTEIS LÍQUIDO
           (igual ao seu modelo: dias_uteis_analise)
           ======================================================= */
        CALCULATED dias_uteis_analise AS dias_uteis_liquido,

        fun.nome AS funci_analise,
        age.nm_super,
        age.nm_regional,
        age.nm_agencia,

        mat.matricula_origem AS matricula_funci_analise,

        /* =======================================================
           8) DIAS ÚTEIS PACTO
           ENTRE:
             > (p.dta_pacto - 15)
             <= p.dta_pacto
           ======================================================= */
        (
          SELECT SUM(c.flg_dia_util)
          FROM work.calendario AS c
          WHERE c.data_ref > (p.dta_pacto - 15)
            AND c.data_ref <= p.dta_pacto
        ) AS dias_uteis_pacto,


        /* =======================================================
           9) DIAS ÚTEIS ETAPA
           ENTRE:
             > p.dta_entrada
             <= (p.dta_saida OU HOJE)
           ======================================================= */
        (
          SELECT SUM(c.flg_dia_util)
          FROM work.calendario AS c
          WHERE c.data_ref > p.dta_entrada
            AND c.data_ref <=
              CASE
                WHEN p.dta_saida IS NULL
                  OR p.dta_saida = 0
                THEN &dt_hoje
                ELSE p.dta_saida
              END
        ) AS dias_uteis_etapa,

        /* =======================================================
           10) DIAS CORRIDOS ETAPA
           ======================================================= */
        (
          CASE
            WHEN p.dta_saida IS NULL
              OR p.dta_saida = 0
            THEN &dt_hoje
            ELSE p.dta_saida
          END
          -
          p.dta_entrada
        ) AS dias_corridos_etapa,

        /* =======================================================
           FLAGS DE FAIXA 
        ======================================================= */
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 0  AND 4  THEN 1 ELSE 0 END AS "0_4d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 5  AND 9  THEN 1 ELSE 0 END AS "5_9d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 10 AND 14 THEN 1 ELSE 0 END AS "10_14d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 15 AND 19 THEN 1 ELSE 0 END AS "15_19d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 20 AND 24 THEN 1 ELSE 0 END AS "20_24d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 25 AND 29 THEN 1 ELSE 0 END AS "25_29d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 30 AND 39 THEN 1 ELSE 0 END AS "30_39d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 40 AND 49 THEN 1 ELSE 0 END AS "40_49d"n,
        CASE WHEN CALCULATED dias_uteis_dm BETWEEN 50 AND 59 THEN 1 ELSE 0 END AS "50_59d"n,
        CASE WHEN CALCULATED dias_uteis_dm >= 60            THEN 1 ELSE 0 END AS "60_d"n,

        p.protocolo_ano,
        p.protocolo_num

    FROM work.inicial_estoque AS p

    LEFT JOIN
    (
      SELECT lgf.*
      FROM work.logs_gerenciamento_final AS lgf
      GROUP BY
        lgf.cd_cli,
        lgf.ts_aclt_slc,
        lgf.nr_seql_slct,
        lgf.nr_seql_pecs_nvl
      HAVING DATEPART(lgf.dt_saida) = MAX(DATEPART(lgf.dt_saida))
    ) AS lgf
      ON p.cd_cli           = lgf.cd_cli
     AND p.ts_aclt_slc      = lgf.ts_aclt_slc
     AND p.nr_seql_slct     = lgf.nr_seql_slct
     AND p.nr_seql_pecs_nvl = lgf.nr_seql_pecs_nvl

    LEFT JOIN
    (
      SELECT mat.*
      FROM work.finalizados_matricula AS mat
      GROUP BY
        mat.cd_cli,
        mat.ts_aclt_slc,
        mat.nr_seql_slct,
        mat.nr_seql_pecs_nvl
      HAVING mat.matricula_origem = MAX(mat.matricula_origem)
    ) AS mat
      ON p.cd_cli           = mat.cd_cli
     AND p.ts_aclt_slc      = mat.ts_aclt_slc
     AND p.nr_seql_slct     = mat.nr_seql_slct
     AND p.nr_seql_pecs_nvl = mat.nr_seql_pecs_nvl

    LEFT JOIN work.funcionarios AS fun
      ON fun.chave = mat.matricula_origem

    LEFT JOIN work.agencia AS age
      ON STRIP(age.prefixo) = PUT(p.prefixo_agencia, Z4.)

    WHERE lgf.dt_ini_analise IS NOT NULL
  ;
QUIT;

PROC PRINT DATA=work.historico_principal(OBS=3);
RUN;

In [ ]:
/* ============================================================
   1) Conexão (LIBNAME) com Postgres
   ============================================================ */
LIBNAME HV POSTGRES
  SERVER   = "172.16.14.11"
  PORT     = 5432
  DATABASE = "curadoria"
  USER     = "painel_user"
  PASSWORD = "painel_user"
  SCHEMA   = "highvarejo"
  PRESERVE_COL_NAMES = YES
  PRESERVE_TAB_NAMES = YES
  READBUFF = 10000
  CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
;

/* ============================================================
   2) Cria tabela intermediária no WORK removendo colunas
   ============================================================ */
DATA WORK.HISTORICO_PRINCIPAL_STG;
  SET WORK.HISTORICO_PRINCIPAL
    (DROP=
      dias_uteis_pacto
      dias_uteis_etapa
      dias_corridos_etapa

      /* Colunas que começam com número: usar name literal */
      '0_4d'n
      '5_9d'n
      '10_14d'n
      '15_19d'n
      '20_24d'n
      '25_29d'n
      '30_39d'n
      '40_49d'n
      '50_59d'n
      '60_d'n

      protocolo_ano
      protocolo_num
      cd_cli
      ts_aclt_slc
      nr_seql_slct
      nr_seql_pecs_nvl
    );
RUN;

/* ============================================================
   3) Drop da tabela alvo no Postgres
   ============================================================ */
PROC SQL;
  CONNECT TO POSTGRES AS mypg
  ( SERVER="172.16.14.11"
    PORT=5432
    DATABASE="curadoria"
    USER="painel_user"
    PASSWORD="painel_user"
    READBUFF=10000
    CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
  );

  EXECUTE (
    DROP TABLE IF EXISTS highvarejo.tb_historico_principal_geral_62
  ) BY mypg;

  DISCONNECT FROM mypg;
QUIT;

/* ============================================================
   4) Sobe a intermediária para o Postgres (tabela final)
   ============================================================ */
DATA HV.tb_historico_principal_geral_62;
  SET WORK.HISTORICO_PRINCIPAL_STG;
RUN;

/* ============================================================
   5) Remove a tabela intermediária do WORK
   ============================================================ 
PROC DATASETS LIB=WORK NOLIST;
  DELETE HISTORICO_PRINCIPAL_STG;
QUIT;*/

LIBNAME HV CLEAR;

PROC PRINT DATA=WORK.HISTORICO_PRINCIPAL_STG(OBS=3);
RUN;

Obs,id,protocolo,nome_cliente,cod_mci,dta_entrada,dta_saida,dta_pacto,tipo_processo,tipo_produto,tipo_cliente,prefixo_agencia,pilar,unidade_analise,equipe,gerente,gerencia,etapa,tp_finalizacao,dta_registro,dt_ini_analise,dt_fim_analise,dt_ini_ag_analise,dt_fim_ag_analise,dt_ini_diligencia,dt_fim_diligencia,dt_ini_triagem,dt_fim_triagem,dt_ini_comite,dt_fim_comite,dt_ini_esc_superior,dt_fim_esc_superior,dias_uteis_dm,dias_corridos_dm,dias_uteis_triagem,dias_corridos_triagem,dias_uteis_ag_analise,dias_corridos_ag_analise,dias_uteis_analise,dias_corridos_analise,dias_uteis_diligencia,dias_corridos_diligencia,dias_uteis_comite,dias_corridos_comite,dias_uteis_esc_superior,dias_corridos_esc_superior,vlr_limite_atual,vlr_limite_solicitado,oper_estruturada,mercado,processos_sla,pacto_sla,dias_uteis_liquido,funci_analise,nm_super,nm_regional,nm_agencia,matricula_funci_analise
1,0,202500006260,FI MINERACAO LTDA,455152632,14FEB2025,17FEB2025,24FEB2025,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2578,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,14FEB2025,14FEB2025,14FEB2025,.,.,.,.,.,.,14FEB2025,.,.,.,1,3,.,.,.,.,1,3,.,.,1,3,.,.,0.00,,,VAREJO + MPE,NÃO,10,1,,SUPER VAR GRANDE SP,SUP SAO JOSE CAMPOS,SANTA ISABEL,
2,0,202500006273,ECCOMANIA COMERCIO DE COMBUSTIVEIS E DERIVADOS DE PETROLEO L,836879229,14FEB2025,17FEB2025,24FEB2025,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,46,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,14FEB2025,14FEB2025,14FEB2025,.,.,.,.,.,.,14FEB2025,.,.,.,1,3,.,.,.,.,1,3,.,.,1,3,.,.,3500000.00,,,VAREJO + MPE,NÃO,10,1,,SUPER PJ II,SUP PJ C O II,EMPRESA CUIABA,
3,0,202500006259,MADEMILE DERIVADOS DE MADEIRA LTDA,447694844,14FEB2025,17FEB2025,24FEB2025,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2389,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,14FEB2025,14FEB2025,17FEB2025,.,.,.,.,.,.,17FEB2025,.,.,.,1,3,.,.,.,.,1,3,.,.,0,0,.,.,4500000.00,,,VAREJO + MPE,NÃO,10,1,,SUPER VAREJO SC,SUP BLUMENAU,PAPANDUVA,


In [ ]:
PROC SQL;
  /* Entradas por data */
  CREATE TABLE work._entradas AS
  SELECT
      hp.dta_entrada AS data_ref FORMAT=date9.,
      COUNT(DISTINCT hp.protocolo) AS qt_entradas
  FROM work.historico_principal AS hp
  WHERE hp.dta_entrada IS NOT NULL
  GROUP BY hp.dta_entrada
  ;
QUIT;

PROC SQL;
  /* Saídas por data */
  CREATE TABLE work._saidas AS
  SELECT
      hp.dta_saida AS data_ref FORMAT=date9.,
      COUNT(DISTINCT hp.protocolo) AS qt_saidas
  FROM work.historico_principal AS hp
  WHERE hp.dta_saida IS NOT NULL
  GROUP BY hp.dta_saida
  ;
QUIT;

PROC SQL;
  /* Registros por data do datetime */
  CREATE TABLE work._registros AS
  SELECT
      DATEPART(hp.ts_aclt_slc) AS data_ref FORMAT=date9.,
      COUNT(DISTINCT hp.protocolo) AS qt_registros
  FROM work.historico_principal AS hp
  WHERE hp.ts_aclt_slc IS NOT NULL
  GROUP BY CALCULATED data_ref
  ;
QUIT;

PROC SQL;
  /* Status de prazo por data de registro (join por dta_registro) */
  CREATE TABLE work._prazos AS
  SELECT
      hp.dta_registro AS data_ref FORMAT=date9.,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_liquido > hp.dias_uteis_pacto THEN hp.protocolo END) AS qt_fora_pz,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_liquido < hp.dias_uteis_pacto THEN hp.protocolo END) AS qt_antes_pz,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_liquido = hp.dias_uteis_pacto THEN hp.protocolo END) AS qt_no_pz
  FROM work.historico_principal AS hp
  WHERE hp.dta_registro IS NOT NULL
  GROUP BY hp.dta_registro
  ;
QUIT;

PROC SQL;
  /* Consolidar tudo em uma tabela final por data_ref */
  CREATE TABLE work.historico_resumo_final AS
  SELECT
      COALESCE(e.data_ref, s.data_ref, r.data_ref, p.data_ref) AS data_ref FORMAT=date9.,
      COALESCE(e.qt_entradas, 0) AS qt_entradas,
      COALESCE(s.qt_saidas,   0) AS qt_saidas,
      COALESCE(r.qt_registros,0) AS qt_registros,
      COALESCE(p.qt_fora_pz,  0) AS qt_fora_pz,
      COALESCE(p.qt_antes_pz, 0) AS qt_antes_pz,
      COALESCE(p.qt_no_pz,    0) AS qt_no_pz
  FROM work._entradas AS e
  FULL JOIN work._saidas    AS s ON e.data_ref = s.data_ref
  FULL JOIN work._registros AS r ON COALESCE(e.data_ref, s.data_ref) = r.data_ref
  FULL JOIN work._prazos    AS p ON COALESCE(e.data_ref, s.data_ref, r.data_ref) = p.data_ref
  ORDER BY data_ref
  ;
QUIT;

PROC SQL;
  CREATE TABLE work.historico_resultado AS
  SELECT
      /* TODAY() FORMAT=date9. AS dta_estoque,*/
      hp.dta_saida AS dta_estoque,

      /* Dimensões */
      hp.tipo_cliente,
      hp.etapa,
      hp.gerencia,
      hp.unidade_analise,
      hp.equipe,
      hp.gerente,
      hp.pilar,
      hp.tipo_processo,
      hp.nm_super,
      hp.nm_regional,

      /* Medidas de prazo (agregadas para não quebrar o GROUP BY) */
      MAX(hp.dias_uteis_dm)     AS dias_uteis_dm,
      MAX(hp.dias_corridos_dm)  AS dias_corridos_dm,
      MAX(hp.dias_uteis_pacto)  AS dias_uteis_pacto,

      10 AS dias_corridos_pacto, /* REUNIÃO COM LYGIA */

      MAX(hp.dias_uteis_etapa)    AS dias_uteis_etapa,
      MAX(hp.dias_corridos_etapa) AS dias_corridos_etapa,

      /* Faixas (protocolos distintos) - usando flags 0/1 */
      COUNT(DISTINCT CASE WHEN hp.'0_4d'n   = 1 THEN hp.protocolo END) AS qt_00_04,
      COUNT(DISTINCT CASE WHEN hp.'5_9d'n   = 1 THEN hp.protocolo END) AS qt_05_09,
      COUNT(DISTINCT CASE WHEN hp.'10_14d'n = 1 THEN hp.protocolo END) AS qt_10_14,
      COUNT(DISTINCT CASE WHEN hp.'15_19d'n = 1 THEN hp.protocolo END) AS qt_15_19,
      COUNT(DISTINCT CASE WHEN hp.'20_24d'n = 1 THEN hp.protocolo END) AS qt_20_24,
      COUNT(DISTINCT CASE WHEN hp.'25_29d'n = 1 THEN hp.protocolo END) AS qt_25_29,
      COUNT(DISTINCT CASE WHEN hp.'30_39d'n = 1 THEN hp.protocolo END) AS qt_30_39,
      COUNT(DISTINCT CASE WHEN hp.'40_49d'n = 1 THEN hp.protocolo END) AS qt_40_49,
      COUNT(DISTINCT CASE WHEN hp.'50_59d'n = 1 THEN hp.protocolo END) AS qt_50_59,
      COUNT(DISTINCT CASE WHEN hp.'60_d'n   = 1 THEN hp.protocolo END) AS qt_m60,

      /* Métricas vindas do resumo diário (uma linha por data) */
      MAX(hrf.qt_entradas)  AS qt_entadas,
      MAX(hrf.qt_saidas)    AS qt_saidas,
      MAX(hrf.qt_registros) AS qt_estoque,   /* REVISAR */
      MAX(hrf.qt_registros) AS qt_registros,
      MAX(hrf.qt_fora_pz)   AS qt_fora_pz,
      MAX(hrf.qt_antes_pz)  AS qt_antes_pz,
      MAX(hrf.qt_no_pz)     AS qt_no_pz,

      /* Tipo */
      'FINAL' AS tipo_dd length=5,

      /* Tempos por etapa (agregados) */
      MAX(hp.dias_uteis_triagem)          AS dias_uteis_triagem,
      MAX(hp.dias_corridos_triagem)       AS dias_corridos_triagem,
      MAX(hp.dias_uteis_ag_analise)       AS dias_uteis_ag_analise,
      MAX(hp.dias_corridos_ag_analise)    AS dias_corridos_ag_analise,
      MAX(hp.dias_uteis_analise)          AS dias_uteis_analise,
      MAX(hp.dias_corridos_analise)       AS dias_corridos_analise,
      MAX(hp.dias_uteis_diligencia)       AS dias_uteis_diligencia,
      MAX(hp.dias_corridos_diligencia)    AS dias_corridos_diligencia,
      MAX(hp.dias_uteis_comite)           AS dias_uteis_comite,
      MAX(hp.dias_corridos_comite)        AS dias_corridos_comite,
      MAX(hp.dias_uteis_esc_superior)     AS dias_uteis_esc_superior,
      MAX(hp.dias_corridos_esc_superior)  AS dias_corridos_esc_superior,

      /* Contagem de protocolos distintos com tempo > 0 em cada etapa */
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_triagem      > 0 THEN hp.protocolo END) AS qt_triagem,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_ag_analise   > 0 THEN hp.protocolo END) AS qt_ag_analise,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_analise      > 0 THEN hp.protocolo END) AS qt_analise,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_diligencia   > 0 THEN hp.protocolo END) AS qt_diligencia,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_comite       > 0 THEN hp.protocolo END) AS qt_comite,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_esc_superior > 0 THEN hp.protocolo END) AS qt_esc_superior

      /* Data de referência do agrupamento 
      hp.dta_registro FORMAT=date9. AS dta_registro */

  FROM work.historico_principal AS hp

  LEFT JOIN work.historico_resumo_final AS hrf
    ON hrf.data_ref = hp.dta_registro

  GROUP BY
      hp.tipo_cliente,
      hp.etapa,
      hp.gerencia,
      hp.unidade_analise,
      hp.equipe,
      hp.gerente,
      hp.pilar,
      hp.tipo_processo,
      hp.nm_super,
      hp.nm_regional,
      hp.dta_saida
  ;
QUIT;

/* =======================================================
   AJUSTE FINAL:
   - Mantém TODAS as colunas
   - Evita repetição de métricas diárias
   - Não quebra queries posteriores
   ======================================================= */

PROC SORT DATA=work.historico_resultado;
  BY dta_estoque;
RUN;

DATA work.historico_resultado;
  SET work.historico_resultado;
  BY dta_estoque;

  IF NOT FIRST.dta_estoque THEN DO;
    qt_entadas   = .;
    qt_saidas    = .;
    qt_estoque   = .;
    qt_registros = .;
    qt_fora_pz   = .;
    qt_antes_pz  = .;
    qt_no_pz     = .;
  END;
RUN;

proc print data=work.historico_resultado(obs=3);
run;

Obs,dta_estoque,tipo_cliente,etapa,gerencia,unidade_analise,equipe,gerente,pilar,tipo_processo,nm_super,nm_regional,dias_uteis_dm,dias_corridos_dm,dias_uteis_pacto,dias_corridos_pacto,dias_uteis_etapa,dias_corridos_etapa,qt_00_04,qt_05_09,qt_10_14,qt_15_19,qt_20_24,qt_25_29,qt_30_39,qt_40_49,qt_50_59,qt_m60,qt_entadas,qt_saidas,qt_estoque,qt_registros,qt_fora_pz,qt_antes_pz,qt_no_pz,tipo_dd,dias_uteis_triagem,dias_corridos_triagem,dias_uteis_ag_analise,dias_corridos_ag_analise,dias_uteis_analise,dias_corridos_analise,dias_uteis_diligencia,dias_corridos_diligencia,dias_uteis_comite,dias_corridos_comite,dias_uteis_esc_superior,dias_corridos_esc_superior,qt_triagem,qt_ag_analise,qt_analise,qt_diligencia,qt_comite,qt_esc_superior
1,17FEB2025,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ I,SUP PJ PR,1,3,11,10,1,3,3,0,0,0,0,0,0,0,0,0,66,0,146,146,0,66,0,FINAL,.,.,.,.,1,3,.,.,0,0,.,.,0,0,3,0,0,0
2,17FEB2025,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ I,SUP PJ RS,0,0,11,10,1,3,1,0,0,0,0,0,0,0,0,0,.,.,.,.,.,.,.,FINAL,.,.,.,.,1,0,.,.,0,0,.,.,0,0,1,0,0,0
3,17FEB2025,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ II,SUP PJ C O II,1,3,11,10,1,3,1,0,0,0,0,0,0,0,0,0,.,.,.,.,.,.,.,FINAL,.,.,.,.,1,3,.,.,1,3,.,.,0,0,1,0,1,0


PROC SQL;
  /* Entradas por data */
  CREATE TABLE work._entradas AS
  SELECT
      hp.dta_entrada AS data_ref FORMAT=date9.,
      COUNT(DISTINCT hp.protocolo) AS qt_entradas
  FROM work.orico_sorico_principal AS hp
  WHERE hp.dta_entrada IS NOT NULL
  GROUP BY hp.dta_entrada
  ;
QUIT;

PROC SQL;
  /* Saídas por data */
  CREATE TABLE work._saidas AS
  SELECT
      hp.dta_saida AS data_ref FORMAT=date9.,
      COUNT(DISTINCT hp.protocolo) AS qt_saidas
  FROM work.historico_principal AS hp
  WHERE hp.dta_saida IS NOT NULL
  GROUP BY hp.dta_saida
  ;
QUIT;

PROC SQL;
  /* Registros por data do datetime */
  CREATE TABLE work._registros AS
  SELECT
      DATEPART(hp.ts_aclt_slc) AS data_ref FORMAT=date9.,
      COUNT(DISTINCT hp.protocolo) AS qt_registros
  FROM work.historico_principal AS hp
  WHERE hp.ts_aclt_slc IS NOT NULL
  GROUP BY CALCULATED data_ref
  ;
QUIT;

PROC SQL;
  /* Status de prazo por data de registro (join por dta_registro) */
  CREATE TABLE work._prazos AS
  SELECT
      hp.dta_registro AS data_ref FORMAT=date9.,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_liquido > hp.dias_uteis_pacto THEN hp.protocolo END) AS qt_fora_pz,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_liquido < hp.dias_uteis_pacto THEN hp.protocolo END) AS qt_antes_pz,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_liquido = hp.dias_uteis_pacto THEN hp.protocolo END) AS qt_no_pz
  FROM work.historico_principal AS hp
  WHERE hp.dta_registro IS NOT NULL
  GROUP BY hp.dta_registro
  ;
QUIT;

PROC SQL;
  /* Consolidar tudo em uma tabela final por data_ref */
  CREATE TABLE work.historico_resumo_final AS
  SELECT
      COALESCE(e.data_ref, s.data_ref, r.data_ref, p.data_ref) AS data_ref FORMAT=date9.,
      COALESCE(e.qt_entradas, 0) AS qt_entradas,
      COALESCE(s.qt_saidas,   0) AS qt_saidas,
      COALESCE(r.qt_registros,0) AS qt_registros,
      COALESCE(p.qt_fora_pz,  0) AS qt_fora_pz,
      COALESCE(p.qt_antes_pz, 0) AS qt_antes_pz,
      COALESCE(p.qt_no_pz,    0) AS qt_no_pz
  FROM work._entradas AS e
  FULL JOIN work._saidas   AS s ON e.data_ref = s.data_ref
  FULL JOIN work._registros AS r ON COALESCE(e.data_ref, s.data_ref) = r.data_ref
  FULL JOIN work._prazos    AS p ON COALESCE(e.data_ref, s.data_ref, r.data_ref) = p.data_ref
  ORDER BY data_ref
  ;
QUIT;

PROC SQL;
  CREATE TABLE work.historico_resultado AS
  SELECT
      /* TODAY() FORMAT=date9. AS dta_estoque,*/
      hp.dta_saida AS dta_estoque,

      /* Dimensões */
      hp.tipo_cliente,
      hp.etapa,
      hp.gerencia,
      hp.unidade_analise,
      hp.equipe,
      hp.gerente,
      hp.pilar,
      hp.tipo_processo,
      hp.nm_super,
      hp.nm_regional,

      /* Medidas de prazo (agregadas para não quebrar o GROUP BY) */
      MAX(hp.dias_uteis_dm)     AS dias_uteis_dm,
      MAX(hp.dias_corridos_dm)  AS dias_corridos_dm,
      MAX(hp.dias_uteis_pacto)  AS dias_uteis_pacto,

      10 AS dias_corridos_pacto, /* REUNIÃO COM LYGIA */

      MAX(hp.dias_uteis_etapa)    AS dias_uteis_etapa,
      MAX(hp.dias_corridos_etapa) AS dias_corridos_etapa,

      /* Faixas (protocolos distintos) - usando flags 0/1 */
      COUNT(DISTINCT CASE WHEN hp.'0_4d'n   = 1 THEN hp.protocolo END) AS qt_00_04,
      COUNT(DISTINCT CASE WHEN hp.'5_9d'n   = 1 THEN hp.protocolo END) AS qt_05_09,
      COUNT(DISTINCT CASE WHEN hp.'10_14d'n = 1 THEN hp.protocolo END) AS qt_10_14,
      COUNT(DISTINCT CASE WHEN hp.'15_19d'n = 1 THEN hp.protocolo END) AS qt_15_19,
      COUNT(DISTINCT CASE WHEN hp.'20_24d'n = 1 THEN hp.protocolo END) AS qt_20_24,
      COUNT(DISTINCT CASE WHEN hp.'25_29d'n = 1 THEN hp.protocolo END) AS qt_25_29,
      COUNT(DISTINCT CASE WHEN hp.'30_39d'n = 1 THEN hp.protocolo END) AS qt_30_39,
      COUNT(DISTINCT CASE WHEN hp.'40_49d'n = 1 THEN hp.protocolo END) AS qt_40_49,
      COUNT(DISTINCT CASE WHEN hp.'50_59d'n = 1 THEN hp.protocolo END) AS qt_50_59,
      COUNT(DISTINCT CASE WHEN hp.'60_d'n   = 1 THEN hp.protocolo END) AS qt_m60,

      /* Métricas vindas do resumo diário (uma linha por data) */
      MAX(hrf.qt_entradas)  AS qt_entadas,
      MAX(hrf.qt_saidas)    AS qt_saidas,
      MAX(hrf.qt_registros) AS qt_estoque,   /* REVISAR */
      MAX(hrf.qt_registros) AS qt_registros,
      MAX(hrf.qt_fora_pz)   AS qt_fora_pz,
      MAX(hrf.qt_antes_pz)  AS qt_antes_pz,
      MAX(hrf.qt_no_pz)     AS qt_no_pz,

      /* Tipo */
      'FINAL' AS tipo_dd length=5,

      /* Tempos por etapa (agregados) */
      MAX(hp.dias_uteis_triagem)       AS dias_uteis_triagem,
      MAX(hp.dias_corridos_triagem)    AS dias_corridos_triagem,
      MAX(hp.dias_uteis_ag_analise)    AS dias_uteis_ag_analise,
      MAX(hp.dias_corridos_ag_analise) AS dias_corridos_ag_analise,
      MAX(hp.dias_uteis_analise)       AS dias_uteis_analise,
      MAX(hp.dias_corridos_analise)    AS dias_corridos_analise,
      MAX(hp.dias_uteis_diligencia)    AS dias_uteis_diligencia,
      MAX(hp.dias_corridos_diligencia) AS dias_corridos_diligencia,
      MAX(hp.dias_uteis_comite)        AS dias_uteis_comite,
      MAX(hp.dias_corridos_comite)     AS dias_corridos_comite,
      MAX(hp.dias_uteis_esc_superior)  AS dias_uteis_esc_superior,
      MAX(hp.dias_corridos_esc_superior) AS dias_corridos_esc_superior,

      /* Contagem de protocolos distintos com tempo > 0 em cada etapa */
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_triagem      > 0 THEN hp.protocolo END) AS qt_triagem,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_ag_analise   > 0 THEN hp.protocolo END) AS qt_ag_analise,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_analise      > 0 THEN hp.protocolo END) AS qt_analise,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_diligencia   > 0 THEN hp.protocolo END) AS qt_diligencia,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_comite       > 0 THEN hp.protocolo END) AS qt_comite,
      COUNT(DISTINCT CASE WHEN hp.dias_uteis_esc_superior > 0 THEN hp.protocolo END) AS qt_esc_superior

      /* Data de referência do agrupamento 
      hp.dta_registro FORMAT=date9. AS dta_registro */
      
  FROM work.historico_principal AS hp

  LEFT JOIN work.historico_resumo_final AS hrf
    ON hrf.data_ref = hp.dta_registro

  GROUP BY
      hp.tipo_cliente,
      hp.etapa,
      hp.gerencia,
      hp.unidade_analise,
      hp.equipe,
      hp.gerente,
      hp.pilar,
      hp.tipo_processo,
      hp.nm_super,
      hp.nm_regional,
      hp.dta_saida
  ;
QUIT;

proc print data=work.historico_resultado(obs=3);
run;

In [ ]:
LIBNAME HV POSTGRES
  SERVER   = "172.16.14.11"
  PORT     = 5432
  DATABASE = "curadoria"
  USER     = "painel_user"
  PASSWORD = "painel_user"
  SCHEMA   = "highvarejo"
  PRESERVE_COL_NAMES = YES
  PRESERVE_TAB_NAMES = YES
  READBUFF = 10000
  CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
;

/* remove a tabela no Postgres */
PROC SQL;
  CONNECT TO POSTGRES AS mypg
  ( SERVER="172.16.14.11" PORT=5432 DATABASE="curadoria"
    USER="painel_user" PASSWORD="painel_user"
    READBUFF=10000
    CONOPTS="UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
  );

  EXECUTE (
    DROP TABLE IF EXISTS highvarejo.tb_historico_resultado_62
  ) BY mypg;

  DISCONNECT FROM mypg;
QUIT;

/* cria e carrega a tabela no Postgres */
DATA HV.tb_historico_resultado_62;
  SET WORK.HISTORICO_RESULTADO;
RUN;

LIBNAME HV CLEAR;

35                                                         The SAS System                             12:49 Thursday, March 12, 2026

2004       ods listing close;ods html5 (id=saspy_internal) file=_tomods1 options(bitmap_mode='inline') device=svg style=HTMLBlue;
2004     ! ods graphics on / outputfmt=png;
NOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1
2005       
2006       LIBNAME HV POSTGRES
2007         SERVER   = "172.16.14.11"
2008         PORT     = 5432
2009         DATABASE = "curadoria"
2010         USER     = "painel_user"
2011         PASSWORD = XXXXXXXXXXXXX
2012         SCHEMA   = "highvarejo"
2013         PRESERVE_COL_NAMES = YES
2014         PRESERVE_TAB_NAMES = YES
2015         READBUFF = 10000
2016         CONOPTS  = "UseServerSidePrepare=1;UseDeclareFetch=1;Fetch=10000"
2017       ;
NOTE: Libref HV was successfully assigned as follows: 
      Engine:        POSTGRES 
      Physical Name: 172.16.14.11
2018       
2019       /* remove a tabela no Postgres */


In [ ]:
proc contents data=work.historico_principal;
run;


In [ ]:
%%python
import IPython
import pandas as pd

def get_sas_session():
    kernel = IPython.Application.instance().kernel
    return kernel.mva

sas = get_sas_session()

df = sas.sd2df(table="historico_principal", libref="WORK")

df.to_csv(
    "historico_principal.csv",
    index=False,
    sep=";",          
    encoding="utf-8"
)

df.head()

,id,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl,protocolo,nome_cliente,cod_mci,dta_entrada,dta_saida,dta_pacto,tipo_processo,tipo_produto,tipo_cliente,prefixo_agencia,pilar,unidade_analise,equipe,gerente,gerencia,etapa,tp_finalizacao,dta_registro,dt_ini_analise,dt_fim_analise,dt_ini_ag_analise,dt_fim_ag_analise,dt_ini_diligencia,dt_fim_diligencia,dt_ini_triagem,dt_fim_triagem,dt_ini_comite,dt_fim_comite,dt_ini_esc_superior,dt_fim_esc_superior,dias_uteis_dm,dias_corridos_dm,dias_uteis_triagem,dias_corridos_triagem,dias_uteis_ag_analise,dias_corridos_ag_analise,dias_uteis_analise,dias_corridos_analise,dias_uteis_diligencia,dias_corridos_diligencia,dias_uteis_comite,dias_corridos_comite,dias_uteis_esc_superior,dias_corridos_esc_superior,vlr_limite_atual,vlr_limite_solicitado,oper_estruturada,mercado,processos_sla,pacto_sla,dias_uteis_liquido,funci_analise,nm_super,nm_regional,nm_agencia,matricula_funci_analise,dias_uteis_pacto,dias_uteis_etapa,dias_corridos_etapa,0_4d,5_9d,10_14d,15_19d,20_24d,25_29d,30_39d,40_49d,50_59d,60_d,protocolo_ano,protocolo_num
0,0.0,455152632.0,2025-02-11 16:55:00.930075,1.0,2.0,202500006260,FI MINERACAO LTDA,455152632.0,2025-02-14,2025-02-17,2025-02-24,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2578.0,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,2025-02-14,2025-02-14,2025-02-14,NaT,NaT,NaT,NaT,NaT,NaT,2025-02-14,NaT,NaT,NaT,1.0,3.0,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,3.0,NaN,NaN,0.0,NaN,NaN,VAREJO + MPE,NÃO,10.0,1.0,NaN,SUPER VAR GRANDE SP,SUP SAO JOSE CAMPOS,SANTA ISABEL,NaN,11.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6260.0
1,0.0,836879229.0,2025-02-14 11:07:03.696324,1.0,2.0,202500006273,ECCOMANIA COMERCIO DE COMBUSTIVEIS E DERIVADOS...,836879229.0,2025-02-14,2025-02-17,2025-02-24,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,46.0,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,2025-02-14,2025-02-14,2025-02-14,NaT,NaT,NaT,NaT,NaT,NaT,2025-02-14,NaT,NaT,NaT,1.0,3.0,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,3.0,NaN,NaN,3500000.0,NaN,NaN,VAREJO + MPE,NÃO,10.0,1.0,NaN,SUPER PJ II,SUP PJ C O II,EMPRESA CUIABA,NaN,11.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6273.0
2,0.0,447694844.0,2025-02-14 11:03:36.518236,1.0,2.0,202500006259,MADEMILE DERIVADOS DE MADEIRA LTDA,447694844.0,2025-02-14,2025-02-17,2025-02-24,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,2389.0,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,2025-02-14,2025-02-14,2025-02-17,NaT,NaT,NaT,NaT,NaT,NaT,2025-02-17,NaT,NaT,NaT,1.0,3.0,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,NaN,NaN,4500000.0,NaN,NaN,VAREJO + MPE,NÃO,10.0,1.0,NaN,SUPER VAREJO SC,SUP BLUMENAU,PAPANDUVA,NaN,11.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6259.0
3,0.0,204795275.0,2025-02-14 10:49:09.152774,1.0,2.0,202500006242,PALMARES INDUSTRIA METALURGICA LTDA,204795275.0,2025-02-14,2025-02-17,2025-02-24,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,8561.0,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,2025-02-14,2025-02-14,2025-02-17,NaT,NaT,NaT,NaT,NaT,NaT,2025-02-17,NaT,NaT,NaT,1.0,3.0,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,NaN,NaN,1000000.0,NaN,NaN,VAREJO + MPE,NÃO,10.0,1.0,NaN,SUPER PJ I,SUP PJ PR,EMPRESA PARANAENSE,NaN,11.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2025.0,6242.0
4,0.0,31965140.0,2025-02-14 10:46:13.147654,1.0,2.0,202500006238,NATURAL BRANDS LTDA,31965140.0,2025-02-14,2025-02-17,2025-02-24,REVISÃO,LIMITE DE CRÉDITO-PJ-METOD.ANALITICA,P. JURÍDICA,1187.0,VAREJO,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,GEREM,FINALIZADO,CONCLUÍDO,2025-02-14,2025-02-14,2025-02-17,NaT,NaT,NaT,NaT,NaT,NaT,2025-02-17,NaT,NaT,NaT,1.0,3.0,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,NaN,NaN,7000000.0,NaN,NaN,VAREJO + MPE,NÃO,10.0,1.0,NaN,SUPER PJ I,SUP PJ PR,EMPRESA MARINGA,NaN,11.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0

In [ ]:
%%python
import IPython
import pandas as pd

def get_sas_session():
    kernel = IPython.Application.instance().kernel
    return kernel.mva

sas = get_sas_session()

df = sas.sd2df(table="logs_gerenciamento_final", libref="WORK")

df.to_csv(
    "logs_gerenciamento_final.csv",
    index=False,
    sep=";",          
    encoding="utf-8"
)

df.head()

,cd_cli,ts_aclt_slc,nr_seql_slct,nr_seql_pecs_nvl,dt_entrada,dt_ini_comite,dt_ini_comite_segundo_voto,dt_finalizacao,dt_saida,dt_ini_analise,dt_devolucao,dt_cancelamento
0,7023.0,2025-02-25 13:16:18.300712,1.0,2.0,2025-02-26 10:03:24.384840,2025-02-26 11:28:14.139799,2025-02-27 10:18:29.802828,NaT,2025-02-27 18:01:14.625138,2025-02-26 10:03:24.384840,NaT,NaT
1,7023.0,2025-02-25 13:16:18.300712,2.0,2.0,2025-03-28 14:51:14.965495,2025-03-28 16:24:05.942024,2025-03-28 18:03:38.042674,NaT,2025-03-31 09:01:06.918926,2025-03-28 16:00:10.111777,NaT,NaT
2,7069.0,2026-03-05 11:56:23.014712,1.0,2.0,2026-03-05 17:31:02.303430,2026-03-09 09:05:49.344306,2026-03-09 13:46:32.171109,NaT,2026-03-10 18:01:14.371697,2026-03-09 08:10:43.929212,NaT,NaT
3,7069.0,2026-03-05 11:56:23.014712,2.0,2.0,2026-03-12 08:49:28.766013,2026-03-12 10:30:35.459503,NaT,NaT,NaT,2026-03-12 09:03:12.012276,NaT,NaT
4,7195.0,2025-04-01 11:12:16.015481,1.0,2.0,2025-04-01 13:41:11.704751,2025-04-02 11:16:28.075724,2025-04-02 14:09:11.554174,NaT,2025-04-02 17:30:54.796741,2025-04-01 13:41:21.661869,NaT,NaT


In [ ]:
%%python
import IPython
import pandas as pd

def get_sas_session():
    kernel = IPython.Application.instance().kernel
    return kernel.mva

sas = get_sas_session()

df = sas.sd2df(table="historico_resultado", libref="WORK")

df.to_csv(
    "historico_resultado.csv",
    index=False,
    sep=";",          
    encoding="utf-8"
)

df.head()

,dta_estoque,tipo_cliente,etapa,gerencia,unidade_analise,equipe,gerente,pilar,tipo_processo,nm_super,nm_regional,dias_uteis_dm,dias_corridos_dm,dias_uteis_pacto,dias_corridos_pacto,dias_uteis_etapa,dias_corridos_etapa,qt_00_04,qt_05_09,qt_10_14,qt_15_19,qt_20_24,qt_25_29,qt_30_39,qt_40_49,qt_50_59,qt_m60,qt_entadas,qt_saidas,qt_estoque,qt_registros,qt_fora_pz,qt_antes_pz,qt_no_pz,tipo_dd,dias_uteis_triagem,dias_corridos_triagem,dias_uteis_ag_analise,dias_corridos_ag_analise,dias_uteis_analise,dias_corridos_analise,dias_uteis_diligencia,dias_corridos_diligencia,dias_uteis_comite,dias_corridos_comite,dias_uteis_esc_superior,dias_corridos_esc_superior,qt_triagem,qt_ag_analise,qt_analise,qt_diligencia,qt_comite,qt_esc_superior
0,2025-02-17,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ I,SUP PJ PR,1.0,3.0,11.0,10.0,1.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66.0,0.0,146.0,146.0,0.0,66.0,0.0,FINAL,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,3.0,0.0,0.0,0.0
1,2025-02-17,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ I,SUP PJ RS,0.0,0.0,11.0,10.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FINAL,NaN,NaN,NaN,NaN,1.0,0.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0
2,2025-02-17,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ II,SUP PJ C O II,1.0,3.0,11.0,10.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FINAL,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,1.0,0.0,1.0,0.0
3,2025-02-17,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER PJ II,SUP PJ NORTE,1.0,3.0,11.0,10.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FINAL,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0
4,2025-02-17,P. JURÍDICA,FINALIZADO,GEREM,GEREM-SP ANALÍTICO,GEREM-SP DIV 5 PJ 62,ALBANO CARVALHO,VAREJO,REVISÃO,SUPER VAR GRANDE SP,SUP SAO JOSE CAMPOS,1.0,3.0,11.0,10.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FINAL,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,1.0,3.0,NaN,NaN,0.0,0.0,1.0,0.0,1.0,0.0


In [ ]:
%%python
import IPython
import pandas as pd

def get_sas_session():
    kernel = IPython.Application.instance().kernel
    return kernel.mva

sas = get_sas_session()

df = sas.sd2df(table="historico_resumo_final", libref="WORK")

df.to_csv(
    "historico_resumo_final.csv",
    index=False,
    sep=";",          
    encoding="utf-8"
)

df.head()

,data_ref,qt_entradas,qt_saidas,qt_registros,qt_fora_pz,qt_antes_pz,qt_no_pz
0,2024-09-30,0.0,0.0,1.0,0.0,0.0,0.0
1,2024-10-01,0.0,0.0,1.0,0.0,0.0,0.0
2,2024-10-02,0.0,0.0,2.0,0.0,0.0,0.0
3,2024-10-07,0.0,0.0,5.0,0.0,0.0,0.0
4,2024-10-08,0.0,0.0,1.0,0.0,0.0,0.0


In [23]:
%%python

import nbformat
from nbconvert import ScriptExporter

with open("tb_estoque_principal_geral_62.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

exporter = ScriptExporter()
text, _ = exporter.from_notebook_node(nb)

with open("arquivo.txt", "w", encoding="utf-8") as f:
    f.write(text)